# 2026 Hungarian Grand Prix: A Reproducible Story-Discovery Audit

## Objective

The purpose of this notebook is to search the 2026 Hungarian Grand Prix race for an observational story that is:

- quantitatively interesting;
- robust enough to survive basic validation and sensitivity checks;
- relevant to knowledgeable Formula 1 readers;
- not already obvious from standard timing tools, telemetry viewers, final-results pages, or ordinary race reports.

The analysis does not assume that a sufficiently strong standalone story must
exist. A finding is only promoted when the available evidence supports it.
Reaching the conclusion that no candidate is strong enough for a substantive
standalone race article is also considered a valid outcome.

## What this notebook covers

The workflow includes:

1. environment and dependency inspection;
2. FastF1 and OpenF1 data provenance;
3. persistent FastF1 cache configuration;
4. race-session data loading and coverage checks;
5. a full data-quality assessment;
6. explicit lap-eligibility and event-labeling rules;
7. broad lap-level and stint-level story discovery;
8. candidate generation and editorial screening;
9. targeted validation of the strongest remaining pace candidate;
10. a bounded sector-level follow-up comparison;
11. sensitivity checks and final editorial closure.

The notebook uses FastF1 lap, stint, sector, position, track-status, and
race-control data, together with preserved OpenF1 records, the official FIA
Final Race Classification, and a limited FIA race-report reference.

Telemetry and weather data are not loaded because none of the retained
findings requires a reliable corner-level or weather-based explanation.

## Final outcome

The analysis retained two secondary findings and rejected one initially
promising performance candidate:

- the difference between the physical finishing order and the official
  classification after two unserved five-second penalties;
- a repeatable Sector 2 timing difference between Charles Leclerc and Lewis
  Hamilton during their final stint;
- an extended-stint pace pattern involving Max Verstappen that was tested in
  Stage 6 but did not survive targeted validation as a main finding.

The penalty reconstruction was correct but routine. The Leclerc-Hamilton
Sector 2 pattern was repeatable, but the available evidence did not establish
a verified mechanism or meaningful on-track consequence. Verstappen's final
soft stint showed relatively low lap-time variation, but the available
comparisons did not establish exceptional tyre preservation, superior
late-race pace, or a better alternative strategy.

None of these findings met the predefined threshold for the substantive
standalone Hungarian Grand Prix race story originally sought.

The final editorial decision is:

`NO MAIN-ARTICLE-STRENGTH STORY IDENTIFIED`

This does not mean that the project failed or that the notebook contains no
useful material. The completed notebook remains a reproducible
story-discovery audit containing valid secondary observations, rejected
hypotheses, documented limitations, and lessons for future race selection.


## Execution note

Read and run the notebook from top to bottom because later cells depend on objects created earlier. Some FastF1 cells may take time to process even when the required data are already available in the local cache.

All analyses through Stage 7 rely on lap-level timing, sector, stint, position, track-status, and race-control records. Telemetry and weather are not loaded.


## Environment and dependency check

Before loading any race data, I record the software environment used to run this notebook. The cell below prints the Python version, operating system, FastF1 version, key scientific Python package versions, Jupyter kernel version, and the Python executable name.

This information helps make the analysis easier to reproduce and troubleshoot, especially because library behavior can change across versions. The cell only inspects the current environment and does not load race data or modify any files.

In [1]:
import sys
import platform
from pathlib import Path

import fastf1
import pandas as pd
import numpy as np
import matplotlib
import IPython
import ipykernel

print("Python        :", sys.version.replace("\n", " "))
print("Platform      :", platform.platform())
print("FastF1        :", fastf1.__version__)
print("pandas        :", pd.__version__)
print("NumPy         :", np.__version__)
print("Matplotlib    :", matplotlib.__version__)
print("IPython       :", IPython.__version__)
print("ipykernel     :", ipykernel.__version__)
print("Executable    :", Path(sys.executable).name)


Python        : 3.11.15 | packaged by conda-forge | (main, Mar  5 2026, 16:59:26) [Clang 19.1.7 ]
Platform      : macOS-26.3-arm64-arm-64bit
FastF1        : 3.8.3
pandas        : 2.3.3
NumPy         : 2.4.4
Matplotlib    : 3.10.9
IPython       : 9.13.0
ipykernel     : 7.2.0
Executable    : python


## Understanding the data sources

FastF1 does not provide raw telemetry taken directly from the teams. It retrieves and processes data published through Formula 1's public live-timing system.

It is useful to distinguish four levels of information used in this notebook.

### Source data

FastF1 provides access to records such as:

- session and driver information;
- lap and sector times;
- lap timestamps;
- track-status information;
- pit entry and exit records;
- tyre compound, tyre life, and stint information;
- race-control messages;
- car and positional telemetry channels.

These records originate from public timing feeds rather than private team systems.

### FastF1 processing

FastF1 parses the timing feeds and organizes them into tables and telemetry objects that are easier to analyze. Some information may be aligned, reconciled, corrected, resampled, or merged by FastF1 before it appears in the notebook.

For example, telemetry channels are not necessarily supplied as one perfectly synchronized table. FastF1 performs processing to combine them into a usable form.

### Calculations made in this notebook

Values such as pace differences, paired sector deltas, stint summaries, position changes, and sensitivity-test results are calculated locally in this notebook.

They are not official Formula 1 values and are not supplied directly by FastF1. Exploratory calculations are treated as diagnostics until they have been checked against the relevant controls, alternative specifications, and supporting evidence.

### Analytical interpretation

Any explanation of what a numerical pattern might mean is an interpretation made in this notebook. Observed values, derived calculations, and possible explanations are kept separate so that an interpretation is not presented as an established fact without sufficient evidence.

## Evidence hierarchy

When sources disagree, this project gives priority to:

1. official FIA documents;
2. structured timing sources such as FastF1, OpenF1, and Jolpica;
3. external reporting and other processed-data tools used for contextual verification.

FastF1, OpenF1, and Jolpica should not be treated as completely independent confirmations because they may ultimately rely on the same official timing lineage.  

## External evidence manifest

The notebook's executable calculations use FastF1 data downloaded at runtime.
The following non-FastF1 evidence supports contextual checks:

- FIA Final Race Classification, Document 62:
  [official FIA PDF](https://www.fia.com/system/files/decision-document/2026_hungarian_grand_prix_-_final_race_classification.pdf)
  (accessed 2026-07-30). The PDF is linked rather than redistributed.
- Preserved OpenF1 VSC and position observations:
  `evidence/openf1_hungary_2026_race_control.json` and
  `evidence/openf1_hungary_2026_position.json`.
- OpenF1 provenance:
  `evidence/README.md`, recording the endpoint, exact query parameters,
  UTC retrieval timestamp, and SHA-256 checksum for each file.

The FastF1 calculations run independently of these evidence files.
OpenF1-derived statements are independently reproducible only when both
repository files are committed. Any material disagreement between sources
must be reported explicitly rather than resolved silently.


## FastF1 cache configuration

Before loading the session, I enable FastF1's local cache. The cache stores downloaded timing responses so that later runs can reuse them instead of requesting the same data again.

The cache is an operational dependency rather than a project result. It is stored outside the repository and is not tracked by Git.

In [2]:
import os
from pathlib import Path

default_cache_root = Path(
    os.environ.get("XDG_CACHE_HOME", Path.home() / ".cache")
)

FASTF1_CACHE_DIR = Path(
    os.environ.get(
        "FASTF1_CACHE",
        default_cache_root / "fastf1" / "Hungarian_GP_2026",
    )
).expanduser()

FASTF1_CACHE_DIR.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(FASTF1_CACHE_DIR))

cache_source = (
    "FASTF1 environment variable"
    if os.environ.get("FASTF1_CACHE")
    else "per-user cache default"
)
print("FastF1 cache configured:", cache_source)


FastF1 cache configured: per-user cache default


## Initial bounded session load

The next cell loads the race-session data needed for the first stages of the analysis while deliberately excluding telemetry and weather.

At this point in the workflow, the notebook loads:

- lap and sector timing;
- session and driver information;
- session results as exposed by FastF1;
- stint, tyre compound, tyre life, pit-entry, and pit-exit fields where available;
- track and session status;
- race-control messages.

Car telemetry and weather data are excluded because they are not required for
the bounded analyses retained in this notebook. Later stages continue to use
lap-level timing, sector, stint, position, track-status, and race-control
records rather than loading telemetry or weather. 

On the first execution, FastF1 may download the required timing data and store them in the configured local cache. On later runs, previously cached files can be reused, although FastF1 may still need time to parse and assemble the session data.

Run this cell and allow it to finish before continuing. The following sections depend on the loaded `session` object.

In [3]:
import datetime as dt

print(f"[{dt.datetime.now().isoformat(timespec='seconds')}] Requesting session object: 2026 Hungary Race...")
session = fastf1.get_session(2026, "Hungary", "R")
print(f"[{dt.datetime.now().isoformat(timespec='seconds')}] Session object created: {session}")

print(
    f"[{dt.datetime.now().isoformat(timespec='seconds')}] "
    "Loading laps, session/driver info, results, and race-control "
    "messages (telemetry and weather excluded)..."
)
session.load(
    laps=True,
    telemetry=False,
    weather=False,
    messages=True,
)
print(f"[{dt.datetime.now().isoformat(timespec='seconds')}] Load complete.")

SESSION_LOAD_TIMESTAMP = dt.datetime.now().isoformat(timespec="seconds")
print("Recorded session-load timestamp:", SESSION_LOAD_TIMESTAMP)


[2026-07-30T22:31:15] Requesting session object: 2026 Hungary Race...


core           INFO 	Loading data for Hungarian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


[2026-07-30T22:31:16] Session object created: 2026 Season Round 11: Hungarian Grand Prix - Race
[2026-07-30T22:31:16] Loading laps, session/driver info, results, and race-control messages (telemetry and weather excluded)...


core        WARNING 	Fixed incorrect tyre stint information for driver '11'
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '12', '16', '44', '6', '63', '30', '27', '41', '5', '10', '18', '14', '43', '31', '23', '55', '87', '81', '11', '77']


[2026-07-30T22:31:16] Load complete.
Recorded session-load timestamp: 2026-07-30T22:31:16


## Initial load verification

Before proceeding with the analysis, I check that the session was loaded successfully and that the main FastF1 tables are available.

The next cell reports:

- event and session information;
- the dimensions of the lap, result, track-status, and race-control tables;
- the drivers represented in the lap data;
- the columns available in the lap and result tables.

This is a structural sanity check rather than a complete data-quality assessment. It confirms that the load produced non-empty data with the expected coverage and schema. The following stages examine missing values, duplicate records, FastF1 quality flags, timing completeness, and analytical eligibility.

In [4]:
print("Event                        :", session.event["EventName"], "-", session.event["EventDate"])
print("Session                      :", session.name)
print("Session date                 :", getattr(session, "date", None))
print()
print("laps.shape                   :", session.laps.shape)
print("results.shape                :", session.results.shape if session.results is not None else None)
print("track_status.shape           :", session.track_status.shape if session.track_status is not None else None)
print("race_control_messages.shape  :", session.race_control_messages.shape if session.race_control_messages is not None else None)
print()

drivers_in_laps = (
    sorted(session.laps["Driver"].dropna().unique().tolist())
    if "Driver" in session.laps.columns
    else []
)
print(f"Distinct drivers in laps ({len(drivers_in_laps)}):", drivers_in_laps)
print()

print("laps.columns:")
print(list(session.laps.columns))
print()

if session.results is not None:
    print("results.columns:")
    print(list(session.results.columns))
else:
    print("results: not available")

Event                        : Hungarian Grand Prix - 2026-07-26 00:00:00
Session                      : Race
Session date                 : 2026-07-26 13:00:00

laps.shape                   : (1431, 31)
results.shape                : (22, 22)
track_status.shape           : (12, 3)
race_control_messages.shape  : (80, 9)

Distinct drivers in laps (22): ['ALB', 'ALO', 'ANT', 'BEA', 'BOR', 'BOT', 'COL', 'GAS', 'HAD', 'HAM', 'HUL', 'LAW', 'LEC', 'LIN', 'NOR', 'OCO', 'PER', 'PIA', 'RUS', 'SAI', 'STR', 'VER']

laps.columns:
['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate']

results.columns:
['DriverNumber', 'Broadca

### Initial load verification result

The session load completed successfully.

The lap table contains 1,431 records across all 22 race participants. The FastF1 results table also contains 22 drivers, while the track-status and race-control tables are both populated.

The available lap columns include the timing, sector, stint, tyre, pit, position, track-status, deletion, and accuracy fields required for the later stages of the analysis.

This confirms that the session data have sufficient structural coverage to proceed. It does not yet establish that every record is analytically valid. The following stages examine missing values, duplicate records, FastF1 quality flags, timing completeness, deleted-lap handling, and analytical eligibility.

# Stage 2: Full lap-level data-quality assessment

Before using the session data for story discovery, this stage checks:

- lap-record coverage;
- duplicate driver and lap-number combinations;
- driver-identifier consistency;
- missing values;
- FastF1 accuracy, deletion, and generated-lap flags;
- track-status and race-control coverage;
- tyre, stint, pit, and position fields;
- result-table semantics.

The purpose is not to fill or repair data silently. It is to document the observed data quality, identify limitations, and determine which discovery modules can proceed and under what restrictions.

Telemetry and weather remain outside this stage.


In [5]:
# Create a working copy so later transformations do not modify session.laps.
laps = session.laps.copy()

print("Working lap table created:", laps.shape)


Working lap table created: (1431, 31)


In [6]:
# Coverage, duplicates, and driver-identifier consistency

lap_counts = (
    laps.groupby("Driver")
    .agg(
        RecordedLaps=("LapNumber", "size"),
        MaxLapNumber=("LapNumber", "max"),
    )
)

results_idx = session.results.set_index("Abbreviation")

coverage = lap_counts.join(
    results_idx[["Laps", "ClassifiedPosition", "Status", "Position"]]
)

coverage = coverage.rename(
    columns={
        "Laps": "ResultsLaps",
        "Position": "ClassifiedPositionNum",
    }
)

coverage["RecordedMinusResultsLaps"] = (
    coverage["RecordedLaps"] - coverage["ResultsLaps"]
)

print("Lap-record coverage compared with the FastF1 result table:")
print(coverage.sort_values("ClassifiedPositionNum").to_string())
print()

dup_count = int(
    laps.duplicated(subset=["Driver", "LapNumber"]).sum()
)
print("Duplicate (Driver, LapNumber) records:", dup_count)

driver_number_counts = laps.groupby("Driver")["DriverNumber"].nunique()
inconsistent_ids = driver_number_counts[driver_number_counts > 1]

print(
    "Drivers associated with more than one DriverNumber:",
    inconsistent_ids.to_dict(),
)

coverage_exceptions = coverage.loc[
    coverage["RecordedMinusResultsLaps"] != 0
]

print()
print("Drivers with a lap-record coverage difference:")
print(
    coverage_exceptions[
        [
            "RecordedLaps",
            "ResultsLaps",
            "RecordedMinusResultsLaps",
            "Status",
        ]
    ].to_string()
)


Lap-record coverage compared with the FastF1 result table:
        RecordedLaps  MaxLapNumber  ResultsLaps ClassifiedPosition    Status  ClassifiedPositionNum  RecordedMinusResultsLaps
Driver                                                                                                                       
NOR               70          70.0         70.0                  1  Finished                    1.0                       0.0
VER               70          70.0         70.0                  2  Finished                    2.0                       0.0
ANT               70          70.0         70.0                  3  Finished                    3.0                       0.0
LEC               70          70.0         70.0                  4  Finished                    4.0                       0.0
HAM               70          70.0         70.0                  5  Finished                    5.0                       0.0
HAD               70          70.0         70.0            

In [7]:
coverage_difference_drivers = coverage_exceptions.index.tolist()

final_extra_rows = (
    laps.loc[
        laps["Driver"].isin(coverage_difference_drivers),
        [
            "Driver",
            "LapNumber",
            "LapTime",
            "Sector1Time",
            "Sector2Time",
            "Sector3Time",
            "PitInTime",
            "PitOutTime",
            "TrackStatus",
            "IsAccurate",
            "FastF1Generated",
        ],
    ]
    .sort_values(["Driver", "LapNumber"])
    .groupby("Driver", group_keys=False)
    .tail(2)
)

print(final_extra_rows.to_string(index=False))

Driver  LapNumber                LapTime            Sector1Time            Sector2Time            Sector3Time              PitInTime PitOutTime TrackStatus  IsAccurate  FastF1Generated
   BOT       13.0 0 days 00:01:29.621000 0 days 00:00:30.918000 0 days 00:00:33.119000 0 days 00:00:25.584000                    NaT        NaT           1        True            False
   BOT       14.0                    NaT 0 days 00:00:31.423000 0 days 00:00:47.298000                    NaT 0 days 01:15:52.545000        NaT           1       False            False
   PIA       55.0 0 days 00:01:24.596000 0 days 00:00:29.497000 0 days 00:00:30.874000 0 days 00:00:24.225000                    NaT        NaT           1        True            False
   PIA       56.0                    NaT                    NaT                    NaT                    NaT                    NaT        NaT        1267       False             True


### Coverage interpretation

All 22 race participants are represented in the lap table. No duplicate
`Driver` and `LapNumber` combinations or inconsistent driver-number mappings
were found.

For Bottas and Piastri, the lap table contains one more row than the completed
lap count reported in the FastF1 results table.

Bottas's lap 14 is an incomplete final lap record. It contains Sector 1 and
Sector 2 times and a pit-entry timestamp, but no Sector 3 time or completed
`LapTime`.

Piastri's lap 56 is also incomplete. It contains no lap or sector times and is
marked by FastF1 as a generated record.

Both rows lack a completed `LapTime` and are marked `IsAccurate=False`.
They therefore do not represent additional completed race laps. They are retained and disclosed rather than
treated as duplicates or silently removed. Their exclusion from analytical
pace comparisons is handled later through the explicit lap-eligibility rules.

In [8]:
# Missingness across timing, sector, tyre, stint, pit, position, and quality fields

missing_cols = [
    "LapTime",
    "Sector1Time",
    "Sector2Time",
    "Sector3Time",
    "Sector1SessionTime",
    "Sector2SessionTime",
    "Sector3SessionTime",
    "SpeedI1",
    "SpeedI2",
    "SpeedFL",
    "SpeedST",
    "Compound",
    "TyreLife",
    "FreshTyre",
    "Stint",
    "PitOutTime",
    "PitInTime",
    "Position",
    "TrackStatus",
    "Deleted",
    "DeletedReason",
    "FastF1Generated",
    "IsAccurate",
    "LapStartDate",
]

missing_from_schema = sorted(set(missing_cols) - set(laps.columns))
assert not missing_from_schema, (
    f"Expected lap columns are missing: {missing_from_schema}"
)

missingness = pd.DataFrame(
    {
        "missing_count": laps[missing_cols].isna().sum(),
        "missing_pct": (
            laps[missing_cols].isna().mean() * 100
        ).round(1),
    }
).sort_values("missing_count", ascending=False)

print(missingness.to_string())
print()

pit_timestamp_mask = (
    laps["PitInTime"].notna()
    | laps["PitOutTime"].notna()
)

print(
    "Laps containing a pit-in or pit-out timestamp:",
    f"{int(pit_timestamp_mask.sum())} / {len(laps)}",
)

print(
    "Missing LapStartDate values:",
    f"{int(laps['LapStartDate'].isna().sum())} / {len(laps)}",
)


                    missing_count  missing_pct
LapStartDate                 1431        100.0
PitOutTime                   1385         96.8
PitInTime                    1384         96.7
SpeedFL                        48          3.4
TyreLife                       25          1.7
Sector1SessionTime             25          1.7
Sector1Time                    23          1.6
LapTime                         2          0.1
Sector3SessionTime              2          0.1
SpeedST                         2          0.1
Sector3Time                     2          0.1
DeletedReason                   1          0.1
Position                        1          0.1
Sector2Time                     1          0.1
Sector2SessionTime              1          0.1
SpeedI2                         1          0.1
SpeedI1                         1          0.1
FreshTyre                       0          0.0
Compound                        0          0.0
TrackStatus                     0          0.0
Deleted      

### Missingness interpretation

Not every missing value represents a data-quality defect.

`PitInTime` and `PitOutTime` are event-specific fields, so they are expected to
be missing on ordinary laps without a pit entry or exit. In this session, 93
of the 1,431 lap records contain at least one pit timestamp.

The two missing `LapTime` values correspond to the incomplete final records
for Bottas and Piastri identified in the coverage check above. These records
are not treated as completed race laps.

Missingness in the sector and speed-trap fields is limited. These values are
not filled or inferred silently. Any analysis that depends on a particular
sector or speed field excludes records where that required value is missing.

`TyreLife` is missing for 25 lap records, or 1.7% of the lap table. Tyre-age
analysis therefore uses only records with an observed tyre-life value and
reports that restriction explicitly.

`Compound`, `Stint`, `TrackStatus`, `Deleted`, `FastF1Generated`, and
`IsAccurate` are populated for every lap record, providing the fields needed
for the later eligibility rules.

`LapStartDate` is unavailable for all records in this initial load. Because
telemetry was not loaded at this stage, the lap table does not provide native
per-lap wall-clock timestamps. No precise wall-clock alignment is inferred
from this field in Stage 2.

In [9]:
# FastF1 quality and processing flags

accurate_counts = laps["IsAccurate"].value_counts(dropna=False)
print("IsAccurate value counts:")
print(accurate_counts.to_string())
print()

generated_mask = (
    laps["FastF1Generated"]
    .fillna(False)
    .astype(bool)
)

print("FastF1Generated value counts:")
print(laps["FastF1Generated"].value_counts(dropna=False).to_string())
print()

print("FastF1-generated lap records:")
print(
    laps.loc[
        generated_mask,
        ["Driver", "LapNumber"],
    ].to_string(index=False)
)
print()

deleted_mask = (
    laps["Deleted"]
    .fillna(False)
    .astype(bool)
)

print("Deleted lap count:", int(deleted_mask.sum()))
print()

print("Deleted lap records:")
print(
    laps.loc[
        deleted_mask,
        ["Driver", "LapNumber", "LapTime", "DeletedReason"],
    ].to_string(index=False)
)

IsAccurate value counts:
IsAccurate
True     1295
False     136

FastF1Generated value counts:
FastF1Generated
False    1430
True        1

FastF1-generated lap records:
Driver  LapNumber
   PIA       56.0

Deleted lap count: 16

Deleted lap records:
Driver  LapNumber                LapTime                   DeletedReason
   NOR       29.0 0 days 00:01:26.546000  TRACK LIMITS AT TURN 4 LAP 29 
   NOR       60.0 0 days 00:01:23.616000  TRACK LIMITS AT TURN 9 LAP 60 
   GAS       38.0 0 days 00:01:29.252000  TRACK LIMITS AT TURN 7 LAP 38 
   ANT       19.0 0 days 00:01:25.632000  TRACK LIMITS AT TURN 7 LAP 19 
   ANT       24.0 0 days 00:01:24.119000  TRACK LIMITS AT TURN 7 LAP 24 
   ALO       26.0 0 days 00:01:29.776000 TRACK LIMITS AT TURN 13 LAP 26 
   ALB       12.0 0 days 00:01:31.197000  TRACK LIMITS AT TURN 4 LAP 12 
   ALB       15.0 0 days 00:01:28.500000  TRACK LIMITS AT TURN 7 LAP 15 
   ALB       16.0 0 days 00:01:28.309000  TRACK LIMITS AT TURN 7 LAP 16 
   LIN       17.0 0

### Quality-flag interpretation

FastF1 marks 1,295 lap records as accurate and 136 as inaccurate.

An `IsAccurate=False` value does not necessarily mean that the underlying row
is corrupted. It indicates that FastF1 does not consider the lap fully suitable
for reliable lap-time analysis, for example because the lap is incomplete,
affected by pit activity, or contains other timing irregularities.

These records remain in the original lap table for transparency. Later pace and
sector comparisons use explicit Stage 3 eligibility rules that exclude
inaccurate laps where required.

Only one record is marked `FastF1Generated=True`: Piastri's lap 56. This is the
incomplete final record already identified in the coverage check.

Sixteen laps are marked as deleted. All 16 deletion reasons relate to track
limits. Their recorded times remain visible in the data, but they are excluded
from analyses that require valid competitive lap or sector times.

### Track-status interpretation

FastF1 stores the track-status codes that applied during each lap in the
`TrackStatus` field. When the status changes during a lap, more than one code
may appear in the same string.

For example, `1267` indicates that codes 1, 2, 6, and 7 applied at different
points during that lap. It should not be interpreted as one four-digit status
code.

The codes observed in this race are:

- `1`: all clear;
- `2`: yellow flag;
- `6`: Virtual Safety Car deployed;
- `7`: Virtual Safety Car ending.

Combined values such as `12`, `21`, `71`, `126`, `1267`, `2671`, and `671`
represent laps that overlapped one or more status transitions.

No Safety Car or red-flag status appears in the observed session-level
track-status table.

In [10]:
# Track-status coverage

observed_track_status = sorted(
    laps["TrackStatus"].astype(str).unique().tolist()
)

print(
    "Distinct per-lap TrackStatus strings:",
    observed_track_status,
)
print()

print("Session-level track-status events:")
print(session.track_status.to_string(index=False))

Distinct per-lap TrackStatus strings: ['1', '12', '126', '1267', '21', '2671', '671', '71']

Session-level track-status events:
                  Time Status     Message
       0 days 00:00:00      2      Yellow
0 days 00:06:07.109000      1    AllClear
0 days 00:14:56.422000      2      Yellow
0 days 00:21:37.077000      1    AllClear
0 days 01:49:26.645000      2      Yellow
0 days 01:49:29.699000      1    AllClear
0 days 02:05:28.377000      2      Yellow
0 days 02:05:30.893000      1    AllClear
0 days 02:13:30.752000      2      Yellow
0 days 02:13:45.164000      6 VSCDeployed
0 days 02:15:03.072000      7   VSCEnding
0 days 02:15:14.197000      1    AllClear


In [11]:
# Race-control message coverage

rcm = session.race_control_messages

print(
    "Race-control message categories:",
    rcm["Category"].value_counts(dropna=False).to_dict(),
)
print(
    "Race-control flags:",
    rcm["Flag"].value_counts(dropna=False).to_dict(),
)
print(
    "Race-control scopes:",
    rcm["Scope"].value_counts(dropna=False).to_dict(),
)
print()

preferred_columns = [
    "Time",
    "Lap",
    "Category",
    "Message",
    "Flag",
    "Scope",
    "Sector",
    "RacingNumber",
]

display_columns = [
    column for column in preferred_columns
    if column in rcm.columns
]

print("SafetyCar-category messages:")
print(
    rcm.loc[
        rcm["Category"] == "SafetyCar",
        display_columns,
    ].to_string(index=False)
)
print()

print("Blue-flag messages:")
print(
    rcm.loc[
        rcm["Flag"] == "BLUE",
        display_columns,
    ].to_string(index=False)
)

Race-control message categories: {'Other': 50, 'Flag': 28, 'SafetyCar': 2}
Race-control flags: {None: 52, 'BLUE': 11, 'CLEAR': 6, 'YELLOW': 5, 'GREEN': 2, 'DOUBLE YELLOW': 2, 'BLACK AND WHITE': 1, 'CHEQUERED': 1}
Race-control scopes: {None: 52, 'Sector': 12, 'Driver': 12, 'Track': 4}

SafetyCar-category messages:
               Time  Lap  Category      Message Flag Scope  Sector RacingNumber
2026-07-26 14:22:55   56 SafetyCar VSC DEPLOYED None  None     NaN         None
2026-07-26 14:24:13   57 SafetyCar   VSC ENDING None  None     NaN         None

Blue-flag messages:
               Time  Lap Category                                            Message Flag  Scope  Sector RacingNumber
2026-07-26 13:15:08    9     Flag WAVED BLUE FLAG FOR CAR 87 (BEA) TIMED AT 15:15:07 BLUE Driver     NaN           87
2026-07-26 13:15:12    9     Flag WAVED BLUE FLAG FOR CAR 18 (STR) TIMED AT 15:15:11 BLUE Driver     NaN           18
2026-07-26 13:15:19    9     Flag WAVED BLUE FLAG FOR CAR 77 (BOT) TIM

### Race-control message interpretation

The race-control table contains 80 messages. Of these, 50 are categorized as
`Other`, 28 as `Flag`, and two as `SafetyCar`.

The `Flag` and `Scope` fields are category-dependent. Their `None` values are
therefore expected for messages that do not use those fields and should not be
treated as missing-data defects.

Blue flags are the most frequent named flag, appearing in 11 message records.
These messages concern nine different cars because car 11, Sergio Perez,
received three separate blue-flag messages. The message count therefore should
not be interpreted as 11 unique drivers or 11 completed lapping events.

The two records categorized as `SafetyCar` explicitly state `VSC DEPLOYED` and
`VSC ENDING`, on laps 56 and 57 respectively. They represent a Virtual Safety
Car sequence, not a full Safety Car period. This is consistent with the
session-level track-status table.

The observed message scopes include driver-specific, sector-specific, and
track-wide notices. Messages without a scope are mainly records for which that
field is not applicable.

This confirms that race-control information is available for later event
labeling and cross-checking. Individual incidents are interpreted using their
message text, timestamps, lap numbers, and associated car numbers rather than
aggregate counts alone.

### FastF1 processing warning

During the initial session load, FastF1 reported:

`Fixed incorrect tyre stint information for driver '11'`

This correction was applied internally by FastF1 while processing the data for
car 11, Sergio Perez. The notebook did not manually alter the affected stint
information.

The warning is documented because tyre and stint fields are used later in the
analysis.

### Result-table semantics

`session.results["Position"]` contains the final classified order exposed by
FastF1, including applied time penalties. It is different from the physical
running order recorded lap by lap in `laps["Position"]`.

Material conclusions involving the final classification are checked against
the official FIA Final Race Classification, Document 62.

The result table contains three status categories: `Finished`, `Lapped`, and
`Retired`. In this notebook, a classified finisher includes both `Finished`
and `Lapped` drivers.

In [12]:
# Result-table coverage and status categories

result_view = session.results[
    [
        "Abbreviation",
        "Position",
        "ClassifiedPosition",
        "Status",
        "Laps",
    ]
].sort_values("Position")

print("FastF1 result table:")
print(result_view.to_string(index=False))
print()

observed_statuses = sorted(
    session.results["Status"].dropna().unique().tolist()
)

print("Distinct result Status values:", observed_statuses)

FastF1 result table:
Abbreviation  Position ClassifiedPosition   Status  Laps
         NOR       1.0                  1 Finished  70.0
         VER       2.0                  2 Finished  70.0
         ANT       3.0                  3 Finished  70.0
         LEC       4.0                  4 Finished  70.0
         HAM       5.0                  5 Finished  70.0
         HAD       6.0                  6 Finished  70.0
         RUS       7.0                  7 Finished  70.0
         LAW       8.0                  8   Lapped  69.0
         HUL       9.0                  9   Lapped  69.0
         LIN      10.0                 10   Lapped  69.0
         BOR      11.0                 11   Lapped  69.0
         GAS      12.0                 12   Lapped  69.0
         STR      13.0                 13   Lapped  69.0
         ALO      14.0                 14   Lapped  69.0
         COL      15.0                 15   Lapped  68.0
         OCO      16.0                 16   Lapped  68.0
         A

### Result-table interpretation

The FastF1 result table contains all 22 race participants:

- seven drivers are recorded as `Finished`;
- twelve are recorded as `Lapped`;
- three are recorded as `Retired`.

For retired drivers, `Position` still provides a numerical ordering, while
`ClassifiedPosition` is recorded as `R`. A numerical `Position` therefore does
not by itself mean that a driver was classified as a finisher.

In the later analysis, classified finishers are identified using
`Status != "Retired"`, which includes both `Finished` and `Lapped` drivers.

The `Position` field represents the final order exposed by FastF1 and is
separate from the lap-by-lap running order in `laps["Position"]`. Any material
claim involving the final classification is cross-checked against the official
FIA Final Race Classification, Document 62.

### Stage 2 verdict

| Discovery module | Proceed? | Required restriction |
|---|---:|---|
| Physical order versus final classification | Yes | Compare lap-level running order with the final classification and verify material conclusions against FIA Document 62 and `evidence/openf1_hungary_2026_race_control.json`, `evidence/openf1_hungary_2026_position.json`, and their provenance manifest. |
| Stint-relative pace | Yes | Apply the Stage 3 eligibility rules before comparing pace. |
| Tyre-age and pace retention | Yes | Use only records with observed `TyreLife` and disclose the 25 missing values. |
| Sector and track-region analysis | Restricted | Use sector analysis only for one bounded pairwise follow-up justified independently of the sector deltas, not as an unrestricted search for isolated differences. |
| Pit and neutralization sequence | Yes | Do not assume that every pit timestamp has the same entry, exit, or stationary-stop meaning. |
| Position and pace mismatch | Yes | Use eligible laps rather than naive full-race averages and include both finished and lapped classified drivers. |
| Timing and classification discontinuities | Yes | Cross-check material conclusions against FIA and `evidence/openf1_hungary_2026_race_control.json`, `evidence/openf1_hungary_2026_position.json`, and their provenance manifest. |

### Stage 2 conclusion

The session data are sufficiently complete to proceed with all seven discovery
modules, although sector analysis remains restricted and the Stage 3
eligibility rules must be applied before pace comparisons are made.

The quality assessment does not declare every lap usable. It documents the
available evidence, identifies known limitations, and defines the restrictions
that the next stage converts into explicit lap-level eligibility flags.

# Stage 3: Explicit lap eligibility and event labeling

Stage 2 established that the session data are sufficiently complete for
story discovery, provided that known timing, pit, track-status, and quality
limitations are handled explicitly.

This stage adds Boolean labels to a copy of the lap table. No original row is
deleted or modified in `laps`.

The labels identify:

- opening laps;
- pit-entry and pit-exit laps;
- laps affected by yellow flags, Virtual Safety Car periods, Safety Car
  periods, or red flags;
- inaccurate, deleted, generated, or incomplete records;
- laps belonging to drivers who later retired;
- laps that satisfy the general eligibility rule for green-flag pace analysis.

No lap-time outlier threshold is introduced here. Eligibility is based only on
documented event, completeness, and FastF1 quality fields.

In [13]:
# Create the Stage 3 working table and event labels.

required_stage3_columns = [
    "Driver",
    "LapNumber",
    "LapTime",
    "Sector1Time",
    "Sector2Time",
    "Sector3Time",
    "PitInTime",
    "PitOutTime",
    "TrackStatus",
    "Position",
    "IsAccurate",
    "Deleted",
    "FastF1Generated",
]

missing_stage3_columns = sorted(
    set(required_stage3_columns) - set(laps.columns)
)

assert not missing_stage3_columns, (
    f"Stage 3 requires missing columns: {missing_stage3_columns}"
)

flagged = laps.copy()

# Race-phase and pit labels
flagged["IsOpeningLap"] = flagged["LapNumber"].eq(1)
flagged["IsPitInLap"] = flagged["PitInTime"].notna()
flagged["IsPitOutLap"] = flagged["PitOutTime"].notna()
flagged["HasPitActivity"] = (
    flagged["IsPitInLap"]
    | flagged["IsPitOutLap"]
)

# Track-status labels
track_status = (
    flagged["TrackStatus"]
    .fillna("")
    .astype(str)
)

flagged["IsVSCLap"] = (
    track_status.str.contains("6", regex=False)
    | track_status.str.contains("7", regex=False)
)

flagged["IsSafetyCarLap"] = track_status.str.contains(
    "4",
    regex=False,
)

flagged["IsRedFlagLap"] = track_status.str.contains(
    "5",
    regex=False,
)

flagged["IsYellowFlagLap"] = track_status.str.contains(
    "2",
    regex=False,
)

flagged["IsPureGreenFlagLap"] = track_status.eq("1")

flagged["IsYellowOrAbnormalLap"] = (
    ~flagged["IsPureGreenFlagLap"]
)

### Track-status labeling

A lap is treated as fully green only when its complete `TrackStatus` value is
`1`, meaning all clear throughout the lap.

When the track status changes during a lap, FastF1 may store several codes in
the same string. For example, `1267` indicates that all-clear, yellow, VSC
deployment, and VSC-ending conditions occurred at different points during
that lap.

The individual status labels preserve information about which conditions
occurred. The broader `IsYellowOrAbnormalLap` label identifies every lap whose
status is not exactly `1`.

This rule does not estimate how much of a lap was affected. A lap that overlaps
even a short status transition is conservatively excluded from the general
green-flag pace sample.

In [14]:
# Quality, completeness, and driver-status labels

is_accurate = (
    flagged["IsAccurate"]
    .fillna(False)
    .astype(bool)
)

is_deleted = (
    flagged["Deleted"]
    .fillna(False)
    .astype(bool)
)

is_generated = (
    flagged["FastF1Generated"]
    .fillna(False)
    .astype(bool)
)

flagged["IsInaccurate"] = ~is_accurate
flagged["IsDeletedLap"] = is_deleted
flagged["IsFastF1Generated"] = is_generated

flagged["HasMissingLapTime"] = flagged["LapTime"].isna()

flagged["HasMissingSectorData"] = (
    flagged[
        [
            "Sector1Time",
            "Sector2Time",
            "Sector3Time",
        ]
    ]
    .isna()
    .any(axis=1)
)

flagged["HasMissingPosition"] = (
    flagged["Position"].isna()
)

flagged["IsIncompleteLapRecord"] = (
    flagged["HasMissingLapTime"]
)

retired_drivers = set(
    session.results.loc[
        session.results["Status"] == "Retired",
        "Abbreviation",
    ]
)

flagged["IsRetiredDriverLap"] = (
    flagged["Driver"].isin(retired_drivers)
)

### General green-flag eligibility rule

A lap is considered an eligible green-flag lap when all of the following are
true:

1. it is not the opening lap;
2. it has no pit-entry or pit-exit timestamp;
3. its complete track-status value is `1`;
4. FastF1 marks it as accurate;
5. it is not marked as deleted;
6. it is not a FastF1-generated record;
7. it has a completed `LapTime`.

This is a general pace-analysis eligibility rule. It does not guarantee that
every field required by every later module is available.

For example:

- sector analysis must additionally require complete sector times;
- tyre-age analysis must additionally require an observed `TyreLife`;
- retirement status is not itself an exclusion because valid pre-retirement
  laps may still be analytically useful.

In [15]:
# General green-flag eligibility

flagged["IsEligibleGreenFlagLap"] = (
    ~flagged["IsOpeningLap"]
    & ~flagged["HasPitActivity"]
    & flagged["IsPureGreenFlagLap"]
    & ~flagged["IsInaccurate"]
    & ~flagged["IsDeletedLap"]
    & ~flagged["IsFastF1Generated"]
    & ~flagged["HasMissingLapTime"]
)

flag_cols = [
    "IsOpeningLap",
    "IsPitInLap",
    "IsPitOutLap",
    "HasPitActivity",
    "IsPureGreenFlagLap",
    "IsYellowFlagLap",
    "IsVSCLap",
    "IsSafetyCarLap",
    "IsRedFlagLap",
    "IsYellowOrAbnormalLap",
    "IsInaccurate",
    "IsDeletedLap",
    "IsFastF1Generated",
    "HasMissingLapTime",
    "HasMissingSectorData",
    "HasMissingPosition",
    "IsIncompleteLapRecord",
    "IsRetiredDriverLap",
    "IsEligibleGreenFlagLap",
]

flag_counts = (
    flagged[flag_cols]
    .sum()
    .rename("lap_count")
    .to_frame()
)

flag_counts["pct_of_all_laps"] = (
    flag_counts["lap_count"]
    / len(flagged)
    * 100
).round(1)

print("Stage 3 lap-label counts:")
print(flag_counts.to_string())
print()

eligible_count = int(
    flagged["IsEligibleGreenFlagLap"].sum()
)

print(
    "Eligible green-flag laps:",
    f"{eligible_count} / {len(flagged)}",
)

Stage 3 lap-label counts:
                        lap_count  pct_of_all_laps
IsOpeningLap                   22              1.5
IsPitInLap                     47              3.3
IsPitOutLap                    46              3.2
HasPitActivity                 93              6.5
IsPureGreenFlagLap           1347             94.1
IsYellowFlagLap                67              4.7
IsVSCLap                       37              2.6
IsSafetyCarLap                  0              0.0
IsRedFlagLap                    0              0.0
IsYellowOrAbnormalLap          84              5.9
IsInaccurate                  136              9.5
IsDeletedLap                   16              1.1
IsFastF1Generated               1              0.1
HasMissingLapTime               2              0.1
HasMissingSectorData           24              1.7
HasMissingPosition              1              0.1
IsIncompleteLapRecord           2              0.1
IsRetiredDriverLap            118              8.2
IsEli

In [16]:
# Verify that eligible laps satisfy every stated criterion.

eligible_rows = flagged.loc[
    flagged["IsEligibleGreenFlagLap"]
]

eligibility_checks = {
    "No opening laps":
        not eligible_rows["IsOpeningLap"].any(),

    "No pit-affected laps":
        not eligible_rows["HasPitActivity"].any(),

    "Only fully green TrackStatus":
        eligible_rows["IsPureGreenFlagLap"].all(),

    "No inaccurate laps":
        not eligible_rows["IsInaccurate"].any(),

    "No deleted laps":
        not eligible_rows["IsDeletedLap"].any(),

    "No FastF1-generated laps":
        not eligible_rows["IsFastF1Generated"].any(),

    "No missing LapTime":
        not eligible_rows["HasMissingLapTime"].any(),
}

eligibility_check_table = pd.Series(
    eligibility_checks,
    name="passed",
).to_frame()

print("Eligibility-rule consistency checks:")
print(eligibility_check_table.to_string())

assert eligibility_check_table["passed"].all(), (
    "At least one eligible lap violates the documented eligibility rule."
)

Eligibility-rule consistency checks:
                              passed
No opening laps                 True
No pit-affected laps            True
Only fully green TrackStatus    True
No inaccurate laps              True
No deleted laps                 True
No FastF1-generated laps        True
No missing LapTime              True


### Eligibility result

The general eligibility rule retains 1,239 of the 1,431 lap records, or 86.6%
of the lap table.

The individual flag counts overlap and should not be added together. For
example, a pit-affected lap may also be marked inaccurate, while an incomplete
generated record may also have a missing lap time.

All eligibility-rule consistency checks pass. The retained sample contains:

- no opening laps;
- no pit-entry or pit-exit laps;
- only laps whose complete `TrackStatus` value is `1`;
- no laps marked inaccurate or deleted;
- no FastF1-generated records;
- no records with a missing `LapTime`.

The retained laps are therefore completed, accurate, undeleted laps that
occurred entirely under all-clear track status and did not contain opening-lap
or pit activity.

This eligibility label provides the common starting sample for later pace
comparisons. It does not establish that the retained laps are traffic-free or
otherwise directly comparable in every respect. Module-specific requirements,
such as complete sector times, observed tyre life, race-phase comparability,
and traffic context, are applied separately where needed.

### Traffic limitation

The initial lap-level FastF1 load does not provide a reliable
gap-to-car-ahead or `DistanceToDriverAhead` field that can be used to identify
traffic consistently across all laps.

`IsEligibleGreenFlagLap` therefore does not mean:

- traffic-free;
- clean air;
- free from traffic effects;
- directly comparable without further context.

It only means that the lap passes the documented timing, pit, track-status,
and quality conditions.

Any later pace comparison that could be influenced by traffic must disclose
that limitation rather than describe the selected laps as clean-air laps.

### Stage 3 conclusion

The original lap table has been preserved, and the Stage 3 working table now
contains explicit event, quality, completeness, and eligibility labels.

`IsEligibleGreenFlagLap` provides a consistent starting sample for the broad
pace analyses in Stage 4. It is not a universal claim that every retained lap
is fully comparable.

Later modules must still apply their own requirements for sector completeness,
tyre-life availability, race phase, stint context, and potential traffic
effects.

# Stage 4: Broad lap-level and stint-level story discovery

This stage uses the labeled lap table from Stage 3 to screen several possible
race stories.

The modules examine:

- differences between the last recorded running order and the final
  classification;
- stint-level pace summaries;
- relationships between tyre life and lap time;
- the pit and position sequence around the Virtual Safety Car;
- mismatches between broad pace rank and finishing position;
- timing and classification discontinuities associated with retirements.

These modules are intended for candidate generation rather than final proof.
Derived values remain exploratory until they survive later relevance,
confounder, sensitivity, and supporting-evidence checks.

No telemetry or weather data are used in this stage. Module D, sector and
track-region analysis, remains restricted and is used later only for one
bounded pairwise follow-up justified using information independent of the
sector results.

## Module A: Last recorded running order versus final classification

For each driver, this module compares:

- the final lap record with a non-missing lap-level `Position`;
- the final classified `Position` exposed by FastF1.

For non-retired drivers, the last recorded position is used as a lap-level
representation of the running order at the finish. It is not treated as an
independent official classification.

The difference is defined as:

`final classified position - last recorded running position`

A positive value means that the driver was classified lower than the last
recorded running position. A negative value means that the driver moved upward
in the final classification.

Material classification differences are checked against the official FIA Final
Race Classification, Document 62, and the available race-control messages. 

A classification difference is not automatically significant enough to become
a main finding. This module is a screening check intended to identify
consequential reorderings, such as changes to the winner, podium, major points
positions, or the interpretation of an important on-track result.

Routine penalty adjustments are retained only as secondary findings unless
they produce a broader sporting or analytical consequence.

In [17]:
# Compare each driver's last recorded running position with the final
# classification exposed by FastF1.

assert flagged["Driver"].nunique() == 22
assert session.results["Abbreviation"].nunique() == 22

result_lookup = session.results.set_index("Abbreviation")

rows = []

for driver, driver_laps in flagged.groupby("Driver", sort=False):
    ordered_laps = driver_laps.sort_values("LapNumber")
    positioned_laps = ordered_laps.dropna(subset=["Position"])

    last_lap_number = (
        positioned_laps["LapNumber"].iloc[-1]
        if not positioned_laps.empty
        else np.nan
    )

    last_running_position = (
        positioned_laps["Position"].iloc[-1]
        if not positioned_laps.empty
        else np.nan
    )

    final_position = result_lookup.loc[driver, "Position"]
    status = result_lookup.loc[driver, "Status"]

    position_delta = (
        final_position - last_running_position
        if pd.notna(last_running_position)
        else np.nan
    )

    rows.append(
        {
            "Driver": driver,
            "LastLapNumber": last_lap_number,
            "LastRunningPosition": last_running_position,
            "FinalClassifiedPosition": final_position,
            "Status": status,
            "PositionDelta": position_delta,
        }
    )

order_check = (
    pd.DataFrame(rows)
    .sort_values("FinalClassifiedPosition")
    .reset_index(drop=True)
)

print("EXPLORATORY DIAGNOSTIC: NOT A RESEARCH FINDING")
print(
    "Last recorded running position compared with the final "
    "classification exposed by FastF1:"
)
print(order_check.to_string(index=False))

EXPLORATORY DIAGNOSTIC: NOT A RESEARCH FINDING
Last recorded running position compared with the final classification exposed by FastF1:
Driver  LastLapNumber  LastRunningPosition  FinalClassifiedPosition   Status  PositionDelta
   NOR           70.0                  1.0                      1.0 Finished            0.0
   VER           70.0                  2.0                      2.0 Finished            0.0
   ANT           70.0                  3.0                      3.0 Finished            0.0
   LEC           70.0                  5.0                      4.0 Finished           -1.0
   HAM           70.0                  4.0                      5.0 Finished            1.0
   HAD           70.0                  6.0                      6.0 Finished            0.0
   RUS           70.0                  7.0                      7.0 Finished            0.0
   LAW           69.0                  8.0                      8.0   Lapped            0.0
   HUL           69.0               

In [18]:
# Inspect non-retired drivers whose final classification differs from their
# last recorded running position.

swaps = order_check.loc[
    order_check["PositionDelta"].ne(0)
    & order_check["Status"].ne("Retired")
].copy()

print(
    "Non-retired drivers with a classification difference:",
    len(swaps),
)
print(swaps.to_string(index=False))
print()

for driver in swaps["Driver"]:
    final_laps = (
        flagged.loc[
            flagged["Driver"].eq(driver),
            [
                "LapNumber",
                "Position",
                "LapTime",
                "TrackStatus",
                "IsAccurate",
                "Deleted",
            ],
        ]
        .sort_values("LapNumber")
        .tail(4)
    )

    print(f"{driver}: final four lap records")
    print(final_laps.to_string(index=False))
    print()

pen_msgs = rcm.loc[
    rcm["Message"].str.contains(
        "PENALTY",
        case=False,
        na=False,
    ),
    ["Time", "Lap", "Message"],
].copy()

print("Penalty-related race-control messages:")
print(pen_msgs.to_string(index=False))

Non-retired drivers with a classification difference: 5
Driver  LastLapNumber  LastRunningPosition  FinalClassifiedPosition   Status  PositionDelta
   LEC           70.0                  5.0                      4.0 Finished           -1.0
   HAM           70.0                  4.0                      5.0 Finished            1.0
   ALB           68.0                 18.0                     17.0   Lapped           -1.0
   SAI           68.0                 19.0                     18.0   Lapped           -1.0
   BEA           68.0                 17.0                     19.0   Lapped            2.0

LEC: final four lap records
 LapNumber  Position                LapTime TrackStatus  IsAccurate  Deleted
      67.0       5.0 0 days 00:01:23.295000           1        True    False
      68.0       5.0 0 days 00:01:22.927000           1        True    False
      69.0       5.0 0 days 00:01:23.003000           1        True    False
      70.0       5.0 0 days 00:01:23.098000           1

### Classification-change interpretation

Five non-retired drivers have different last recorded and final classified
positions:

- Hamilton was running fourth and was classified fifth;
- Leclerc was running fifth and was classified fourth;
- Bearman was running seventeenth and was classified nineteenth;
- Albon moved from eighteenth to seventeenth;
- Sainz moved from nineteenth to eighteenth.

The official FIA Final Race Classification lists unserved five-second
penalties for Hamilton and Bearman. Hamilton's penalty moved Leclerc ahead of
him. Bearman's penalty moved both Albon and Sainz ahead of him.

Race-control messages also record a five-second penalty for Sainz, but that
penalty has a separate `PENALTY SERVED` message. It was therefore already
reflected in the running order and is not one of the two post-race adjustments
responsible for this pattern.

The final lap records show stable running positions through the finish. The
combined evidence therefore supports the following descriptive finding:

> Two unserved five-second penalties caused five classified-position changes
> between the physical finishing order represented by the lap data and the
> official final classification.

This establishes the classification reconstruction. It does not establish a
new technical or strategic mechanism, and it does not imply that five separate
on-track overtakes occurred.

Piastri's much larger position difference is associated with retirement and is
examined separately in Module G.

## Module B: Stint-level eligible-lap pace summaries

This module summarizes eligible green-flag lap times within each combination
of driver, stint, and tyre compound.

For each group, it reports:

- the number of eligible laps;
- the median lap time;
- the lap-time standard deviation;
- the first and last lap numbers represented.

Only groups containing at least four eligible laps are displayed. This is a
reporting threshold intended to avoid ranking groups based on one or two laps.

These summaries remain descriptive. Different stints can occur under different
fuel loads, track conditions, race phases, and traffic situations. A faster
median does not automatically establish superior underlying race pace.

In [19]:
eligible_laps = flagged.loc[
    flagged["IsEligibleGreenFlagLap"]
].copy()

eligible_laps["LapTimeSec"] = (
    eligible_laps["LapTime"].dt.total_seconds()
)

assert len(eligible_laps) == 1239
assert eligible_laps["LapTimeSec"].notna().all()

stint_pace = (
    eligible_laps
    .groupby(
        ["Driver", "Stint", "Compound"],
        dropna=False,
    )
    .agg(
        n=("LapTimeSec", "size"),
        median_s=("LapTimeSec", "median"),
        std_s=("LapTimeSec", "std"),
        first_lap=("LapNumber", "min"),
        last_lap=("LapNumber", "max"),
    )
    .reset_index()
)

stint_pace = (
    stint_pace.loc[stint_pace["n"].ge(4)]
    .sort_values("median_s")
    .reset_index(drop=True)
)

print("EXPLORATORY DIAGNOSTIC: NOT A RESEARCH FINDING")
print(
    "Driver, stint, and compound groups with at least "
    "four eligible green-flag laps:"
)
print()

print("Ten lowest median lap times:")
print(stint_pace.head(10).to_string(index=False))
print()

front_runners = ["NOR", "VER", "ANT", "LEC", "HAM"]

print("Front-runner stint summaries:")
print(
    stint_pace.loc[
        stint_pace["Driver"].isin(front_runners)
    ].to_string(index=False)
)

EXPLORATORY DIAGNOSTIC: NOT A RESEARCH FINDING
Driver, stint, and compound groups with at least four eligible green-flag laps:

Ten lowest median lap times:
Driver  Stint Compound  n  median_s    std_s  first_lap  last_lap
   NOR    4.0     SOFT 12   82.9145 0.391195       58.0      70.0
   LEC    4.0     SOFT 13   82.9270 0.725921       58.0      70.0
   ANT    3.0     HARD 14   83.0085 0.820910       55.0      70.0
   HAM    4.0     SOFT 13   83.2570 0.708276       58.0      70.0
   RUS    3.0     HARD 14   83.3960 0.569396       57.0      70.0
   NOR    3.0     HARD 14   83.8615 0.569725       41.0      55.0
   VER    3.0     SOFT 25   83.9250 0.288328       43.0      70.0
   HAD    3.0     HARD 24   84.1225 0.459909       44.0      70.0
   PIA    3.0     HARD 19   84.4080 0.566681       35.0      55.0
   HAM    3.0     HARD 22   84.4260 0.385190       32.0      55.0

Front-runner stint summaries:
Driver  Stint Compound  n  median_s    std_s  first_lap  last_lap
   NOR    4.0     SO

### Stint-summary interpretation

The lowest median lap times are concentrated in late-race stints. Norris and
Leclerc record the two lowest medians, both during their final soft-tyre
stints, at 82.9145 and 82.9270 seconds respectively.

Antonelli's final hard stint, Hamilton's final soft stint, and Russell's final
hard stint follow in the ranking. Verstappen's final soft stint has a higher
median of 83.9250 seconds, but it covers 25 eligible laps and has relatively
low lap-time variation.

These values should not be treated as a definitive ranking of driver
performance. The groups differ in:

- race phase;
- stint length;
- fuel load;
- tyre compound and tyre history;
- traffic;
- track position.

The concentration of the lowest medians in the closing stages also indicates
that race phase and fuel load are likely contributing to the ranking.

The table is therefore useful as broad screening context, but it does not
establish a sufficiently specific or independently supported finding by
itself. No candidate is promoted solely from this module.

## Module C: Tyre-life and lap-time slope screening

This module fits a simple linear relationship between `TyreLife` and lap time
within each driver, stint, and compound group.

A positive slope means that lap time increased as tyre life increased. A
negative slope means that lap time decreased over the observed portion of the
stint.

Only groups with at least six eligible laps and at least three distinct
tyre-life values are evaluated.

The slope is not treated as a pure tyre-degradation estimate. It can also
reflect:

- fuel burn;
- track evolution;
- traffic;
- driver management;
- race phase;
- changing competitive circumstances.

The calculation is therefore used only as a broad screening diagnostic.

In [20]:
# Summarize the linear relationship between TyreLife and lap time.

tyre_slope_rows = []

for (
    driver,
    stint,
    compound,
), group in eligible_laps.groupby(
    ["Driver", "Stint", "Compound"],
    dropna=False,
):
    valid = (
        group.loc[
            :,
            ["TyreLife", "LapTimeSec"],
        ]
        .dropna()
        .copy()
    )

    if len(valid) < 6:
        continue

    if valid["TyreLife"].nunique() < 3:
        continue

    x = valid["TyreLife"].to_numpy(dtype=float)
    y = valid["LapTimeSec"].to_numpy(dtype=float)

    slope, intercept = np.polyfit(x, y, 1)
    predicted = intercept + slope * x

    residual_sum_squares = np.sum(
        (y - predicted) ** 2
    )

    total_sum_squares = np.sum(
        (y - y.mean()) ** 2
    )

    r_squared = (
        1 - residual_sum_squares / total_sum_squares
        if total_sum_squares > 0
        else np.nan
    )

    tyre_slope_rows.append(
        {
            "Driver": driver,
            "Stint": stint,
            "Compound": compound,
            "n": len(valid),
            "TyreLifeMin": valid["TyreLife"].min(),
            "TyreLifeMax": valid["TyreLife"].max(),
            "LapTimeSlopeSecPerTyreLap": slope,
            "RSquared": r_squared,
        }
    )

deg = (
    pd.DataFrame(tyre_slope_rows)
    .sort_values("LapTimeSlopeSecPerTyreLap")
    .reset_index(drop=True)
)

print("EXPLORATORY DIAGNOSTIC: NOT A RESEARCH FINDING")
print("Number of evaluated stint groups:", len(deg))
print()

print("Five most negative slopes:")
print(deg.head(5).to_string(index=False))
print()

print("Five most positive slopes:")
print(deg.tail(5).to_string(index=False))
print()

print("Front-runner slope summaries:")
print(
    deg.loc[
        deg["Driver"].isin(
            ["NOR", "VER", "ANT", "LEC", "HAM"]
        )
    ].to_string(index=False)
)

EXPLORATORY DIAGNOSTIC: NOT A RESEARCH FINDING
Number of evaluated stint groups: 65

Five most negative slopes:
Driver  Stint Compound  n  TyreLifeMin  TyreLifeMax  LapTimeSlopeSecPerTyreLap  RSquared
   STR    1.0     SOFT  6          2.0          7.0                  -0.307086  0.259739
   RUS    1.0   MEDIUM 25          2.0         26.0                  -0.059148  0.473787
   HUL    2.0     HARD 17          2.0         18.0                  -0.047858  0.246155
   HAM    4.0     SOFT 13          5.0         17.0                  -0.045780  0.063364
   BOR    1.0     SOFT 26          2.0         28.0                  -0.045298  0.385957

Five most positive slopes:
Driver  Stint Compound  n  TyreLifeMin  TyreLifeMax  LapTimeSlopeSecPerTyreLap  RSquared
   HAD    2.0     HARD 20          2.0         22.0                   0.108647  0.601159
   BEA    2.0     HARD 17          2.0         19.0                   0.165101  0.460188
   GAS    4.0     SOFT 13          2.0         14.0        

### Tyre-life screening interpretation

The 65 evaluated stint groups show slopes in both directions. Some lap times
decrease as `TyreLife` increases, while others increase.

The most negative slope belongs to Stroll's first soft-tyre stint, at about
-0.307 seconds per tyre lap. However, it is based on only six eligible laps and
has an `RSquared` of 0.260. `RSquared` measures the proportion of the observed
lap-time variation explained by the fitted linear relationship with
`TyreLife`. A value of 0.260 means that the line explains approximately 26% of
the lap-time variation in this small sample. The remaining variation is not
explained by the fitted line, so this should not be treated as a reliable
isolated pattern.

Among the front runners, most fitted relationships are weak. Several slopes
are close to zero, and many have very low `RSquared` values. For example,
Hamilton's final soft stint and Verstappen's final soft stint both have
negative slopes, but their fitted lines explain only a limited share of the
observed lap-time variation.

Some positive relationships are more pronounced, including Leclerc's second
hard stint and Norris's third hard stint. Even where `RSquared` is higher,
however, the slope cannot be interpreted as tyre degradation alone.

Fuel burn, track evolution, traffic, race phase, driver management, and
changing competitive circumstances remain uncontrolled.

The broad screen therefore identifies patterns that may support a more focused
question, including Verstappen's long final soft stint, but no isolated slope
is promoted as a standalone finding. A candidate still requires a meaningful
race consequence, a suitable comparison, and further validation.

## Why Module D is deferred

Module D covers sector-level and track-region analysis. It is not used as an
unrestricted search across every driver, lap, and sector.

Such a search would create a high risk of selecting an isolated difference
caused by traffic, fuel load, tyre condition, race phase, or ordinary
lap-to-lap variation.

The later Leclerc-Hamilton comparison is bounded using information independent
of the sector deltas:

- Module A identifies them as adjacent physical finishers whose official order
  changes only after Hamilton's penalty;
- Module B shows that both have overlapping final Stint 4 runs on the `SOFT`
  compound;
- the paired design later aligns race-lap number and recorded tyre life.

The follow-up is restricted to this one driver pair and examines all three
sectors rather than searching across the field for the largest sector
difference.

Any resulting sector pattern remains exploratory and can only be retained as a
secondary observation unless a meaningful mechanism or race consequence is
independently established.

## Module E: Pit and position sequence around the Virtual Safety Car

The preserved OpenF1 response documented in `evidence/README.md` places the Virtual Safety Car deployment at
`2026-07-26T14:22:55Z` and the ending message at
`2026-07-26T14:24:13Z`.

They also show that the identity of the driver in second place changed during
Norris's stop sequence:

- Hamilton was the observed second-place driver before the change;
- Verstappen became the observed second-place driver afterward.

The two periods must therefore remain separate. A single before-and-after
calculation against an unspecified "P2" would compare two different drivers.

The FastF1 lap-level records below are used to cross-check the affected laps,
positions, stint changes, and pit timestamps. They are not used to infer the
exact within-lap VSC transition point or the exact instant at which the
second-place driver changed.

In [21]:
vsc_columns = [
    "Driver",
    "LapNumber",
    "Position",
    "LapTime",
    "TrackStatus",
    "PitInTime",
    "PitOutTime",
    "Stint",
    "Compound",
]

vsc_laps = (
    flagged.loc[
        flagged["IsVSCLap"],
        vsc_columns,
    ]
    .sort_values(
        ["LapNumber", "Position"],
        na_position="last",
    )
    .copy()
)

print("EXPLORATORY DIAGNOSTIC: NOT A RESEARCH FINDING")
print("Lap records overlapping VSC deployment or ending:", len(vsc_laps))
print()

vsc_lap_summary = (
    vsc_laps
    .groupby(
        ["LapNumber", "TrackStatus"],
        dropna=False,
    )
    .size()
    .rename("record_count")
    .reset_index()
)

print("VSC-overlap records by lap number and TrackStatus:")
print(vsc_lap_summary.to_string(index=False))
print()

relevant_drivers = ["NOR", "HAM", "VER", "LEC"]

vsc_sequence = (
    flagged.loc[
        flagged["Driver"].isin(relevant_drivers)
        & flagged["LapNumber"].between(55, 58),
        vsc_columns,
    ]
    .sort_values(["Driver", "LapNumber"])
)

print("Norris, Hamilton, Verstappen, and Leclerc, laps 55 to 58:")
print(vsc_sequence.to_string(index=False))

EXPLORATORY DIAGNOSTIC: NOT A RESEARCH FINDING
Lap records overlapping VSC deployment or ending: 37

VSC-overlap records by lap number and TrackStatus:
 LapNumber TrackStatus  record_count
      54.0         126             4
      55.0         126             6
      55.0        1267             2
      55.0        2671             1
      55.0         671             4
      56.0         126             5
      56.0        1267             1
      56.0        2671             1
      56.0         671             6
      56.0          71             2
      57.0         671             5

Norris, Hamilton, Verstappen, and Leclerc, laps 55 to 58:
Driver  LapNumber  Position                LapTime TrackStatus              PitInTime             PitOutTime  Stint Compound
   HAM       55.0       3.0 0 days 00:01:24.403000           1                    NaT                    NaT    3.0     HARD
   HAM       56.0       2.0 0 days 00:01:36.308000         126 0 days 02:14:30.879000          

### VSC-sequence interpretation

Because drivers crossed the timing line at different moments, the VSC-related
status codes appear across driver-specific laps 54 to 57 rather than one
universal lap number. The combined `TrackStatus` values also show that some
laps overlapped transitions into or out of the neutralization period.

The targeted records show that:

- Norris entered the pits on lap 56 and exited on lap 57;
- Hamilton entered the pits on lap 56 and exited on lap 57;
- Leclerc completed the same pit-in and pit-out sequence;
- Verstappen remained on track and was recorded in second place on lap 57.

The preserved OpenF1 samples documented in `evidence/README.md` identify Hamilton as the observed second-place
driver before the relevant change and Verstappen as the observed second-place
driver afterward. A single before-and-after calculation against an unspecified
"P2" would therefore compare two different drivers.

The FastF1 records support the affected pit, stint, and position sequence, but
they do not establish an exact within-lap event boundary or a continuously
sampled margin through the full stop sequence.

This module also does not resolve the previously identified interval-sampling
gap. Missing interval samples are not treated as evidence of a physical event.
The result provides bounded context rather than an independent standalone
finding.

## Module F: Broad pace rank versus final classification

This module compares each classified finisher's median eligible-lap time with
their final classified position.

The measure is intentionally coarse because it pools:

- all stints;
- all compounds;
- different fuel loads;
- different race phases;
- potentially different traffic conditions.

Retired drivers are excluded before pace ranks are calculated so that the pace
rank and classified rank refer to the same group of 19 classified finishers.

A positive `RankMismatch` means the driver was classified lower than their
coarse pace rank. A negative value means the driver was classified higher than
their coarse pace rank.

In [22]:
driver_pace = (
    eligible_laps
    .groupby("Driver")
    .agg(
        EligibleLapCount=("LapTimeSec", "size"),
        MedianEligibleLapSec=("LapTimeSec", "median"),
    )
    .reset_index()
)

pace_classification = driver_pace.merge(
    session.results[
        [
            "Abbreviation",
            "Position",
            "Status",
        ]
    ],
    left_on="Driver",
    right_on="Abbreviation",
    how="left",
    validate="one_to_one",
).drop(columns="Abbreviation")

pace_classification = (
    pace_classification.loc[
        pace_classification["Status"].ne("Retired")
    ]
    .copy()
)

assert len(pace_classification) == 19

pace_classification["PaceRank"] = (
    pace_classification["MedianEligibleLapSec"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

pace_classification["ClassifiedRank"] = (
    pace_classification["Position"].astype(int)
)

pace_classification["RankMismatch"] = (
    pace_classification["ClassifiedRank"]
    - pace_classification["PaceRank"]
)

pace_classification = (
    pace_classification
    .sort_values("ClassifiedRank")
    .reset_index(drop=True)
)

print("EXPLORATORY DIAGNOSTIC: NOT A RESEARCH FINDING")
print(
    "Classified finishers:",
    f"{pace_classification['Status'].eq('Finished').sum()} Finished +",
    f"{pace_classification['Status'].eq('Lapped').sum()} Lapped =",
    len(pace_classification),
)
print()

print(
    pace_classification[
        [
            "Driver",
            "EligibleLapCount",
            "MedianEligibleLapSec",
            "PaceRank",
            "ClassifiedRank",
            "RankMismatch",
            "Status",
        ]
    ].to_string(index=False)
)
print()

largest_mismatches = (
    pace_classification
    .assign(
        AbsoluteMismatch=lambda frame: (
            frame["RankMismatch"].abs()
        )
    )
    .sort_values(
        [
            "AbsoluteMismatch",
            "ClassifiedRank",
        ],
        ascending=[False, True],
    )
    .head(5)
)

print("Five largest absolute rank mismatches:")
print(
    largest_mismatches[
        [
            "Driver",
            "PaceRank",
            "ClassifiedRank",
            "RankMismatch",
        ]
    ].to_string(index=False)
)

EXPLORATORY DIAGNOSTIC: NOT A RESEARCH FINDING
Classified finishers: 7 Finished + 12 Lapped = 19

Driver  EligibleLapCount  MedianEligibleLapSec  PaceRank  ClassifiedRank  RankMismatch   Status
   NOR                60               84.5015         1               1             0 Finished
   VER                61               85.0730         5               2            -3 Finished
   ANT                59               85.0640         4               3            -1 Finished
   LEC                61               84.9200         3               4             1 Finished
   HAM                61               84.7480         2               5             3 Finished
   HAD                61               85.4870         7               6            -1 Finished
   RUS                62               85.2405         6               7             1 Finished
   LAW                60               86.3085         8               8             0   Lapped
   HUL                60              

### Pace-rank interpretation

The 19 classified finishers show generally modest differences between broad
eligible-lap pace rank and final classification rank.

The largest absolute mismatches are:

- Verstappen, who ranked fifth by median eligible-lap time but finished second;
- Hamilton, who ranked second by median eligible-lap time but was classified
  fifth;
- Bortoleto, who ranked thirteenth by pace but finished eleventh;
- Gasly, who ranked tenth by pace but finished twelfth.

These differences are screening signals rather than performance judgments.

A whole-race median pools laps from different stints, compounds, fuel loads,
race phases, traffic conditions, and strategic situations. It also ignores
starting position, incidents, pit-stop timing, tyre allocation, and periods in
which a driver may have been managing rather than attacking.

The table therefore does not show where a driver "deserved" to finish. It only
identifies cases in which broad eligible-lap pace rank differs from final
classification rank.

Verstappen's and Hamilton's three-position mismatches are the largest in the
table, but neither difference has a sufficiently specific and independently
supported mechanism within this module. No mismatch is therefore promoted as
a standalone candidate from this broad ranking screen.

## Module G: Piastri's incomplete final record and retirement

Piastri has a large difference between his last recorded running position and
his final numerical position because he retired while running near the front.

This module inspects:

- his final lap records;
- the quality and generated-record flags;
- race-control messages that directly name car 81;
- race-control context around laps 54 to 57.

The purpose is to determine what the preserved data establish and, equally
importantly, what they do not establish about the retirement.

In [23]:
pia = (
    flagged.loc[
        flagged["Driver"].eq("PIA"),
        [
            "LapNumber",
            "Position",
            "LapTime",
            "Sector1Time",
            "Sector2Time",
            "Sector3Time",
            "TrackStatus",
            "IsAccurate",
            "FastF1Generated",
            "HasMissingLapTime",
        ],
    ]
    .sort_values("LapNumber")
    .tail(6)
)

print("Piastri's final six lap records:")
print(pia.to_string(index=False))
print()

racing_number_numeric = pd.to_numeric(
    rcm["RacingNumber"],
    errors="coerce",
)

pia_rcm = rcm.loc[
    racing_number_numeric.eq(81)
].copy()

print(
    "Race-control messages directly naming car 81:",
    len(pia_rcm),
)

if not pia_rcm.empty:
    print(
        pia_rcm[
            [
                "Time",
                "Lap",
                "Category",
                "Message",
                "Flag",
            ]
        ].to_string(index=False)
    )

print()

context_messages = rcm.loc[
    rcm["Lap"].between(54, 57, inclusive="both"),
    [
        "Time",
        "Lap",
        "Category",
        "Message",
        "Flag",
        "RacingNumber",
    ],
]

print("Race-control context for laps 54 to 57:")
print(context_messages.to_string(index=False))

Piastri's final six lap records:
 LapNumber  Position                LapTime            Sector1Time            Sector2Time            Sector3Time TrackStatus  IsAccurate  FastF1Generated  HasMissingLapTime
      51.0       3.0 0 days 00:01:24.714000 0 days 00:00:29.620000 0 days 00:00:30.825000 0 days 00:00:24.269000           1        True            False              False
      52.0       3.0 0 days 00:01:24.373000 0 days 00:00:29.530000 0 days 00:00:30.739000 0 days 00:00:24.104000           1        True            False              False
      53.0       3.0 0 days 00:01:24.552000 0 days 00:00:29.447000 0 days 00:00:30.842000 0 days 00:00:24.263000           1        True            False              False
      54.0       2.0 0 days 00:01:24.600000 0 days 00:00:29.479000 0 days 00:00:30.972000 0 days 00:00:24.149000           1        True            False              False
      55.0       2.0 0 days 00:01:24.596000 0 days 00:00:29.497000 0 days 00:00:30.874000 0 days 00:00

### Piastri-retirement interpretation

Piastri's completed lap times remain stable at lap-time resolution through lap
55. Across laps 51 to 55, they range from 84.373 to 84.714 seconds, a spread of
0.341 seconds, with no obvious lap-level deterioration before the incomplete
record.

His lap-56 row has no lap time, sector times, or position. It is marked
inaccurate and FastF1-generated, so it represents an incomplete final record
rather than an additional completed lap.

The available race-control table contains no message directly naming car 81.
Therefore, that table alone does not identify the cause of the
neutralization.

However, the
[FIA race recap](https://www.fia.com/news/f1-norris-wins-hungary-piastri-retires-and-verstappen-claims-second-ahead-antonelli)
states that Piastri retired on lap 56 with an apparent gearbox problem and that
the McLaren's stoppage triggered the Virtual Safety Car.  


The surrounding race-control messages are consistent with that sequence:

- a yellow flag in Sector 6;
- double yellow flags;
- marshals on track at Turn 3;
- a Virtual Safety Car deployment and ending.

The combined evidence therefore establishes that Piastri's stopped McLaren
caused the VSC. The FastF1 lap table does not independently diagnose the
mechanical failure, while the FIA race recap describes it as an apparent
gearbox problem.

A nearby stewards message concerning car 18 relates to a yellow-flag
infringement review. Its proximity in the message table is not evidence that
car 18 caused the VSC or that the incident was connected to Piastri's
retirement.

This module reconstructs the retirement and neutralization sequence, but the
sequence is not promoted as a standalone finding because the cause and broad
race consequence were already evident from the live broadcast and official
race reporting.

### Stage 4 conclusion

The broad discovery stage produced one clearly reconstructed but routine
descriptive result:  

- two unserved five-second penalties caused five classified-position changes
  between the last recorded running order and the official final
  classification.

This result is retained for Stage 5 screening, but the reconstruction itself
does not imply sporting importance or originality.  

The other modules provided useful screening evidence:

- stint medians were descriptive but strongly dependent on race phase, fuel
  load, tyre history, traffic, and other uncontrolled conditions;
- tyre-life slopes were confounded and could not be interpreted as pure tyre
  degradation;
- Verstappen's long final soft stint remained a potentially interesting pattern
  requiring a more suitable comparison and further validation;
- the VSC module confirmed the pit and position sequence but did not provide one
  continuous, comparable before-and-after margin because the identity of the
  second-place driver changed;
- broad pace-rank mismatches were too coarse to support performance conclusions;
- Piastri's stoppage was confirmed as the cause of the VSC, while the FIA race
  recap described his retirement as an apparent gearbox problem; this was a
  verified event reconstruction rather than a novel standalone finding.
  
Module D, sector and track-region analysis, was intentionally not used as an
unrestricted discovery screen. It remains reserved for one bounded pairwise
follow-up justified independently of the sector results.

The next stage converts the established finding and the unresolved carried-over
questions into an explicit candidate shortlist.

# Stage 5: Candidate shortlist and disposition

Stage 4 produced one established but routine descriptive result, several
contextual reconstructions, and one unresolved performance pattern. This stage
turns those notebook-supported observations into an explicit shortlist and
decides which candidates should proceed.

Each candidate is assessed qualitatively across:

- evidence status;
- quantitative strength;
- robustness;
- interpretability;
- sporting significance;
- originality or availability elsewhere;
- dependence on an ambiguous event boundary;
- the main unresolved limitation.

No numerical composite score is calculated. A candidate is not advanced merely
because it performs well on one dimension. It must also have a specific
question, defensible evidence, meaningful race relevance, and a realistic path
to resolving its remaining limitations.

Originality and external availability are recorded only where the available
evidence supports a clear judgment. Candidate 1 is retained as a secondary
finding because the five-driver reordering follows directly from the official
penalty and classification records. Candidate 2 is the only candidate carried
forward for targeted validation. The remaining candidates are retained as
verified context or unresolved questions.

In [24]:
shortlist = pd.DataFrame(
    [
        {
            "Candidate": "C1",
            "Title": (
                "Two unserved penalties, five "
                "classification-position changes"
            ),
            "EvidenceStatus": "Established descriptive finding",
            "Question": (
                "How did the unserved five-second penalties for Hamilton "
                "and Bearman change the final classification relative to "
                "the last recorded running order?"
            ),
            "KeyEvidence": (
                "HAM moved from recorded P4 to classified P5; LEC moved "
                "from P5 to P4; BEA moved from P17 to P19; ALB and SAI "
                "each gained one classified position. FIA Document 62 "
                "confirms the unserved penalties for HAM and BEA. SAI's "
                "separate penalty was served and is excluded."
            ),
            "MainLimitation": (
                "The reconstruction is valid, but the changes affected "
                "non-podium positions and may have limited sporting and "
                "editorial significance."
            ),
            "QuantitativeStrength": "High",
            "Robustness": "High",
            "Interpretability": "High",
            "SportingSignificance": "Limited",
            "OriginalityStatus": (
                "Low; directly implied by the official classification "
                "and penalty record"
            ),
            "BoundaryDependence": "Low",
            "Disposition": "Retain as secondary finding only",
        },
        {
            "Candidate": "C2",
            "Title": "Verstappen's long final soft-tyre stint",
            "EvidenceStatus": "Promising but unresolved",
            "Question": (
                "Did Verstappen sustain unusually stable or competitive "
                "pace across his long final soft-tyre stint?"
            ),
            "KeyEvidence": (
                "The stint contains 25 eligible laps, has a median lap "
                "time of 83.925 seconds and a standard deviation of "
                "0.288 seconds. The tyre-life slope is -0.0168 seconds "
                "per tyre lap with an R-squared of 0.253."
            ),
            "MainLimitation": (
                "Fuel burn, track evolution, race phase, traffic, and "
                "different stint timing prevent a fair direct comparison "
                "with Norris, Leclerc, or other front runners."
            ),
            "QuantitativeStrength": "Medium",
            "Robustness": "Medium",
            "Interpretability": "Medium",
            "SportingSignificance": "Potentially meaningful",
            "OriginalityStatus": (
                "Cannot be judged until the pattern is better validated"
            ),
            "BoundaryDependence": "Low",
            "Disposition": "Advance: targeted validation",
        },
        {
            "Candidate": "C3",
            "Title": (
                "Norris's VSC lead margin, separated by "
                "second-place driver"
            ),
            "EvidenceStatus": "Sequence established",
            "Question": (
                "How did Norris's observed lead margin change through his "
                "VSC stop sequence when Hamilton and Verstappen are "
                "treated as separate second-place references?"
            ),
            "KeyEvidence": (
                "The preserved samples show Hamilton as the observed P2 "
                "before the relevant change and Verstappen as the "
                "observed P2 afterward. FastF1 confirms the associated "
                "pit and position sequence."
            ),
            "MainLimitation": (
                "There is no valid continuous before-and-after margin to "
                "one unchanged P2, and the observed compression is "
                "expected during a leader's pit stop."
            ),
            "QuantitativeStrength": "Medium",
            "Robustness": "Medium",
            "Interpretability": "High",
            "SportingSignificance": "Moderate",
            "OriginalityStatus": (
                "Low; the underlying pit-stop effect is expected"
            ),
            "BoundaryDependence": "Medium",
            "Disposition": (
                "Do not advance as a main standalone candidate"
            ),
        },
        {
            "Candidate": "C4",
            "Title": "Front-runner stint-pace hierarchy",
            "EvidenceStatus": "Descriptive only",
            "Question": (
                "Which front-runner stint groups recorded the lowest "
                "median eligible-lap times?"
            ),
            "KeyEvidence": (
                "Norris and Leclerc record the two lowest displayed "
                "medians during their final soft stints. Antonelli, "
                "Hamilton, and Russell follow in the late-race ranking."
            ),
            "MainLimitation": (
                "The ranking compares different race phases, compounds, "
                "stint lengths, fuel loads, traffic conditions, and "
                "competitive situations."
            ),
            "QuantitativeStrength": "Medium",
            "Robustness": "Medium",
            "Interpretability": "High",
            "SportingSignificance": "Moderate",
            "OriginalityStatus": (
                "Low; this is a standard stint-summary analysis"
            ),
            "BoundaryDependence": "Low",
            "Disposition": (
                "Do not advance as a main standalone candidate"
            ),
        },
        {
            "Candidate": "C5",
            "Title": "Piastri's retirement and the Virtual Safety Car",
            "EvidenceStatus": "Established event reconstruction",
            "Question": (
                "What do the preserved timing and external evidence "
                "establish about Piastri's retirement and the VSC?"
            ),
            "KeyEvidence": (
                "Piastri's lap times remain stable through lap 55. His "
                "lap-56 record is incomplete and FastF1-generated. The "
                "FIA race recap identifies his stopped McLaren as the "
                "cause of the VSC and describes an apparent gearbox "
                "problem."
            ),
            "MainLimitation": (
                "The mechanical diagnosis does not come from the FastF1 "
                "lap data, and the event was already visible in the "
                "broadcast and described in official reporting."
            ),
            "QuantitativeStrength": "Low",
            "Robustness": "High",
            "Interpretability": "High",
            "SportingSignificance": "High event significance",
            "OriginalityStatus": (
                "Low; the central event is directly available elsewhere"
            ),
            "BoundaryDependence": "Low",
            "Disposition": "Retain as verified context only",
        },
    ]
)

assert shortlist["Candidate"].is_unique
assert shortlist["Title"].notna().all()
assert shortlist["Disposition"].notna().all()

display_columns = [
    "Candidate",
    "Title",
    "EvidenceStatus",
    "QuantitativeStrength",
    "Robustness",
    "Interpretability",
    "SportingSignificance",
    "OriginalityStatus",
    "BoundaryDependence",
    "Disposition",
]

print("Stage 5 candidate disposition table:")
print(shortlist[display_columns].to_string(index=False))

Stage 5 candidate disposition table:
Candidate                                                        Title                   EvidenceStatus QuantitativeStrength Robustness Interpretability    SportingSignificance                                                       OriginalityStatus BoundaryDependence                                   Disposition
       C1 Two unserved penalties, five classification-position changes  Established descriptive finding                 High       High             High                 Limited Low; directly implied by the official classification and penalty record                Low              Retain as secondary finding only
       C2                      Verstappen's long final soft-tyre stint         Promising but unresolved               Medium     Medium           Medium  Potentially meaningful                  Cannot be judged until the pattern is better validated                Low                  Advance: targeted validation
       C3   Norris's 

In [25]:
# Print the candidate evidence and disposition checks

print("Detailed candidate summaries:")
print()

for row in shortlist.itertuples(index=False):
    print(f"{row.Candidate}: {row.Title}")
    print(f"  Question: {row.Question}")
    print(f"  Evidence status: {row.EvidenceStatus}")
    print(f"  Key evidence: {row.KeyEvidence}")
    print(f"  Main limitation: {row.MainLimitation}")
    print(f"  Disposition: {row.Disposition}")
    print()

advanced_candidates = shortlist.loc[
    shortlist["Disposition"].str.startswith("Advance"),
    ["Candidate", "Title", "Disposition"],
].copy()

print("Candidates advancing beyond Stage 5:")
print(advanced_candidates.to_string(index=False))

assert set(advanced_candidates["Candidate"]) == {"C2"}

Detailed candidate summaries:

C1: Two unserved penalties, five classification-position changes
  Question: How did the unserved five-second penalties for Hamilton and Bearman change the final classification relative to the last recorded running order?
  Evidence status: Established descriptive finding
  Key evidence: HAM moved from recorded P4 to classified P5; LEC moved from P5 to P4; BEA moved from P17 to P19; ALB and SAI each gained one classified position. FIA Document 62 confirms the unserved penalties for HAM and BEA. SAI's separate penalty was served and is excluded.
  Main limitation: The reconstruction is valid, but the changes affected non-podium positions and may have limited sporting and editorial significance.
  Disposition: Retain as secondary finding only

C2: Verstappen's long final soft-tyre stint
  Question: Did Verstappen sustain unusually stable or competitive pace across his long final soft-tyre stint?
  Evidence status: Promising but unresolved
  Key evidence: Th

### Shortlist interpretation

The shortlist retains one candidate for targeted validation. The remaining
observations are classified as secondary findings, verified context, or
unresolved questions.

#### Candidate 1: penalty and classification reconstruction

Candidate 1 is the clearest established descriptive finding from Stage 4. Its
evidence is direct, internally consistent, and supported by the official
classification and penalty records.

However, the resulting position changes are a straightforward consequence of
applying the two unserved penalties to the recorded finishing order. The
reconstruction does not reveal a hidden mechanism or a result that is difficult
to obtain from the official classification.

The changes affected fourth and fifth place, together with positions seventeen
to nineteen. They did not alter the winner or podium.

Candidate 1 is therefore retained as a verified secondary finding and
data-consistency check, but it does not proceed as a main standalone candidate.

#### Candidate 2: Verstappen's final soft stint

Candidate 2 has greater potential sporting interest but weaker comparative
support.

Verstappen completed a 25-lap eligible sample during his final soft-tyre stint,
with a median lap time of 83.925 seconds and a standard deviation of 0.288
seconds. The fitted tyre-life slope was slightly negative, but its
`RSquared` value of 0.253 indicates that tyre life accounted for only a limited
share of the observed lap-time variation.

These values identify a real descriptive pattern, but they do not establish
why it occurred or whether it was unusual after accounting for race phase,
fuel load, track evolution, traffic, and differences in stint timing.

Candidate 2 therefore advances as an unresolved candidate requiring targeted
validation.

#### Remaining candidates

The remaining candidates are not advanced as main standalone stories:

- the Norris VSC sequence compares different second-place drivers and reflects
  an expected pit-stop effect;
- the broad stint hierarchy is descriptive and methodologically conventional;
- Piastri's stoppage and the VSC form a verified race-event reconstruction, but
  the central event was already evident from the broadcast and official
  reporting.

### Stage 5 conclusion

Stage 5 does not select an established main story.

Only one candidate remains active:

- **Candidate 2**, Verstappen's long final soft-tyre stint, advances to
  targeted validation because it has potential sporting interest but still
  lacks a fair comparison and adequate accounting for race phase, fuel load,
  track evolution, and traffic.

Candidate 1, the penalty and classification reconstruction, is retained as a
verified secondary finding. Although correct, it is a straightforward
consequence of the official penalties and final classification rather than an
original analytical result.

The other three candidates are retained only as verified context or unresolved
questions that do not currently justify further analysis.

No telemetry or weather data are loaded in this stage. The next stage focuses
on whether Verstappen's final soft stint survives more targeted comparative
and sensitivity checks.

# Stage 6: Targeted validation of Verstappen's final soft stint

Stage 5 retained only one candidate for further analysis: Verstappen's long
final soft-tyre stint.

The recorded stint spans laps 42 to 70. After applying the exact green-flag
eligibility rule established in Stage 3, 25 laps remain available for analysis.

This stage asks two bounded questions:

1. Does the apparent in-stint lap-time pattern remain similar when the eligible
   sample is divided into pre-VSC and post-VSC portions?
2. What do pairwise comparisons with the late soft-tyre stints of Norris,
   Leclerc, and Hamilton show when either race phase or tyre age is aligned,
   and what remains confounded?

Two complementary cross-driver comparisons are used:

- a same-race-phase comparison using pairwise common eligible lap numbers;
- an exact matched-tyre-age comparison using values common to all four
  eligible samples within the candidate range from 5 to 12.

Neither comparison is assumed to be fully controlled. The same-race-phase
comparison does not match tyre age, while the matched-tyre-age comparison does
not match race phase. Their disagreement or agreement is therefore treated as
a sensitivity result rather than a causal estimate.

The stage reuses `eligible_laps` from Stage 4 so that the filtering rule remains
identical. No telemetry, weather data, external reporting, strategy simulation,
or alternative finishing-position calculation is introduced.

In [26]:
# Examine Verstappen's final soft stint using the existing eligibility rule.

def summarize_linear_trend(label, frame):
    """Return descriptive linear-trend statistics for one lap window."""

    valid = (
        frame.loc[
            :,
            ["LapNumber", "TyreLife", "LapTimeSec"],
        ]
        .dropna()
        .sort_values("LapNumber")
        .copy()
    )

    if len(valid) < 4:
        raise ValueError(
            f"{label} contains fewer than four valid laps."
        )

    if valid["TyreLife"].nunique() < 3:
        raise ValueError(
            f"{label} contains fewer than three tyre-life values."
        )

    x = valid["TyreLife"].to_numpy(dtype=float)
    y = valid["LapTimeSec"].to_numpy(dtype=float)

    slope, intercept = np.polyfit(x, y, 1)
    predicted = intercept + slope * x

    residual_sum_squares = np.sum(
        (y - predicted) ** 2
    )

    total_sum_squares = np.sum(
        (y - y.mean()) ** 2
    )

    r_squared = (
        1 - residual_sum_squares / total_sum_squares
        if total_sum_squares > 0
        else np.nan
    )

    return {
        "Window": label,
        "n": len(valid),
        "LapRange": (
            f"{int(valid['LapNumber'].min())}-"
            f"{int(valid['LapNumber'].max())}"
        ),
        "TyreLifeRange": (
            f"{int(valid['TyreLife'].min())}-"
            f"{int(valid['TyreLife'].max())}"
        ),
        "MedianLapTimeSec": valid["LapTimeSec"].median(),
        "StdSec": valid["LapTimeSec"].std(),
        "SlopeSecPerTyreLap": slope,
        "RSquared": r_squared,
    }


ver_final_soft = (
    eligible_laps.loc[
        eligible_laps["Driver"].eq("VER")
        & eligible_laps["Stint"].eq(3)
        & eligible_laps["Compound"].eq("SOFT")
    ]
    .sort_values("LapNumber")
    .copy()
)

assert len(ver_final_soft) == 25
assert int(ver_final_soft["LapNumber"].min()) == 43
assert int(ver_final_soft["LapNumber"].max()) == 70
assert not ver_final_soft["LapNumber"].isin([56, 57]).any()

ver_pre_vsc = ver_final_soft.loc[
    ver_final_soft["LapNumber"].le(55)
].copy()

ver_post_vsc = ver_final_soft.loc[
    ver_final_soft["LapNumber"].ge(58)
].copy()

stage6_trend_table = pd.DataFrame(
    [
        summarize_linear_trend(
            "Full eligible stint",
            ver_final_soft,
        ),
        summarize_linear_trend(
            "Pre-VSC eligible portion",
            ver_pre_vsc,
        ),
        summarize_linear_trend(
            "Post-VSC eligible portion",
            ver_post_vsc,
        ),
    ]
)

lap_tyre_correlation = ver_final_soft[
    ["LapNumber", "TyreLife"]
].corr().iloc[0, 1]

print(
    "EXPLORATORY VALIDATION DIAGNOSTIC: "
    "NOT A TYRE-DEGRADATION ESTIMATE"
)
print()

print("Verstappen final SOFT-stint trend sensitivity:")
print(
    stage6_trend_table.to_string(
        index=False,
        formatters={
            "MedianLapTimeSec": "{:.4f}".format,
            "StdSec": "{:.4f}".format,
            "SlopeSecPerTyreLap": "{:.4f}".format,
            "RSquared": "{:.4f}".format,
        },
    )
)
print()

print(
    "Correlation between LapNumber and TyreLife "
    f"within the eligible stint: {lap_tyre_correlation:.6f}"
)
print()

print(
    "Because LapNumber and TyreLife advance together within one "
    "continuous stint, the fitted slope cannot independently separate "
    "tyre ageing from fuel burn, track evolution, or other changes "
    "over race time."
)

EXPLORATORY VALIDATION DIAGNOSTIC: NOT A TYRE-DEGRADATION ESTIMATE

Verstappen final SOFT-stint trend sensitivity:
                   Window  n LapRange TyreLifeRange MedianLapTimeSec StdSec SlopeSecPerTyreLap RSquared
      Full eligible stint 25    43-70          5-32          83.9250 0.2883            -0.0168   0.2529
 Pre-VSC eligible portion 12    43-55          5-17          83.9395 0.2154            -0.0349   0.4316
Post-VSC eligible portion 13    58-70         20-32          83.6670 0.3060            -0.0057   0.0052

Correlation between LapNumber and TyreLife within the eligible stint: 1.000000

Because LapNumber and TyreLife advance together within one continuous stint, the fitted slope cannot independently separate tyre ageing from fuel burn, track evolution, or other changes over race time.


## In-stint trend sensitivity interpretation

The fitted trend is not stable across the two race windows.

Across the full eligible stint, lap time decreases by approximately 0.0168
seconds per recorded tyre lap, with an `RSquared` of 0.253. The pre-VSC portion
has a more negative fitted slope of approximately 0.0349 seconds per tyre lap,
and the line accounts for about 43% of the observed variation in that segment.

The post-VSC portion is different. Its fitted slope is approximately 0.0057
seconds per tyre lap in the negative direction, while its `RSquared` is only
0.005. The fitted line therefore accounts for almost none of the post-VSC
lap-time variation.

This sensitivity to the selected window means that the full-stint negative
slope should not be interpreted as one consistent tyre-preservation pattern
across the entire stint.

In addition, the correlation between `LapNumber` and `TyreLife` is exactly
1.000 within the eligible sample. Tyre age and race progression are therefore
inseparable in this single stint. The fitted slopes cannot distinguish tyre
behaviour from fuel burn, track evolution, traffic, driver management, or other
changes over race time.

The in-stint analysis consequently confirms a descriptive pattern but does not
establish unusually low tyre degradation. The next analysis tests how
Verstappen's observed pace compares with other final soft-tyre stints under two
different matching designs.

## Cross-driver sensitivity comparisons

A single cross-driver comparison cannot simultaneously match all relevant
conditions in this case.

Two complementary designs are therefore used. Each one holds one important
dimension constant while leaving another unmatched.

### Comparison A: same race phase

Verstappen is compared pairwise with Norris, Leclerc, and Hamilton on race-lap
numbers from 58 to 70 for which both drivers in each pair have eligible
records.

This approximately aligns race phase and broad track evolution because the
drivers are compared on the same race-lap numbers. It does not align traffic,
track position, or the exact time at which each driver completed the lap.  


However, Verstappen's soft tyres were substantially older than the soft tyres
used by the other three drivers. His median recorded tyre life in these paired
samples is approximately 26 laps, compared with approximately 11 laps for the
other drivers.

This comparison therefore measures the observed late-race lap-time difference.
It is not a controlled estimate of car performance, driver performance, or tyre
degradation because tyre age is not matched.

### Comparison B: exact matched tyre age

The same four drivers are compared at the exact tyre-life values observed for
all four final soft-tyre stints within the candidate range from 5 to 12.

The common observed values are 5, 6, 8, 9, 10, and 11. Tyre-life values 7 and
12 are not included because they are not present in every eligible stint
sample.

This comparison matches recorded tyre age exactly, but it does not match race
phase. Verstappen reaches these tyre ages on laps 43 to 49, while Norris,
Leclerc, and Hamilton reach them on laps 58 to 64. The drivers are therefore observed at different race phases. Verstappen would ordinarily be expected to carry more fuel, while track evolution, traffic
context, and race circumstances may also differ.

The comparison measures the observed lap-time differences at common recorded
tyre ages. It is not a controlled estimate of car performance, driver
performance, or tyre degradation.

### Interpretation boundary

The two comparisons answer different descriptive questions:

- Comparison A asks how Verstappen's observed pace compared with the other
  drivers during the same late-race phase;
- Comparison B asks how his observed pace compared at the same recorded tyre
  ages.

Neither design matches both race phase and tyre age. Traffic is also not
assessed.

Agreement between the two comparisons may strengthen a descriptive observation,
but it cannot establish a causal explanation or isolate tyre behaviour from the
other race-time factors.

In [27]:
# Compare Verstappen with the final SOFT stints of Norris, Leclerc,
# and Hamilton using two complementary matching designs.

final_soft_stints = {
    "VER": 3,
    "NOR": 4,
    "LEC": 4,
    "HAM": 4,
}


def select_final_soft_stint(driver, stint):
    """Select one driver's eligible laps from the specified SOFT stint."""

    selected = (
        eligible_laps.loc[
            eligible_laps["Driver"].eq(driver)
            & eligible_laps["Stint"].eq(stint)
            & eligible_laps["Compound"].eq("SOFT")
        ]
        .sort_values("LapNumber")
        .copy()
    )

    if selected.empty:
        raise ValueError(
            f"No eligible SOFT laps found for {driver}, stint {stint}."
        )

    return selected


final_soft_laps = {
    driver: select_final_soft_stint(driver, stint)
    for driver, stint in final_soft_stints.items()
}

# Comparison A:
# Match the same race laps, while accepting that tyre ages differ.

ver_late = final_soft_laps["VER"].loc[
    final_soft_laps["VER"]["LapNumber"].between(58, 70)
].copy()

same_phase_rows = []

for opponent in ["NOR", "LEC", "HAM"]:
    opponent_late = final_soft_laps[opponent].loc[
        final_soft_laps[opponent]["LapNumber"].between(58, 70)
    ].copy()

    paired = (
        ver_late.loc[
            :,
            ["LapNumber", "LapTimeSec", "TyreLife"],
        ]
        .merge(
            opponent_late.loc[
                :,
                ["LapNumber", "LapTimeSec", "TyreLife"],
            ],
            on="LapNumber",
            suffixes=("_VER", f"_{opponent}"),
            how="inner",
            validate="one_to_one",
        )
        .sort_values("LapNumber")
    )

    if len(paired) < 4:
        raise ValueError(
            f"Too few common late-race laps for VER and {opponent}."
        )

    paired["DeltaSec"] = (
        paired["LapTimeSec_VER"]
        - paired[f"LapTimeSec_{opponent}"]
    )

    same_phase_rows.append(
        {
            "Opponent": opponent,
            "n": len(paired),
            "LapRange": (
                f"{int(paired['LapNumber'].min())}-"
                f"{int(paired['LapNumber'].max())}"
            ),
            "MedianDelta_VER_minus_Opponent_s":
                paired["DeltaSec"].median(),
            "MeanDelta_VER_minus_Opponent_s":
                paired["DeltaSec"].mean(),
            "VERFasterLaps":
                int(paired["DeltaSec"].lt(0).sum()),
            "OpponentFasterLaps":
                int(paired["DeltaSec"].gt(0).sum()),
            "MedianVERTyreLife":
                paired["TyreLife_VER"].median(),
            "MedianOpponentTyreLife":
                paired[f"TyreLife_{opponent}"].median(),
        }
    )

same_phase_comparison = pd.DataFrame(same_phase_rows)

print(
    "Comparison A: same race phase and common lap numbers, "
    "but different tyre ages"
)
print(
    same_phase_comparison.to_string(
        index=False,
        formatters={
            "MedianDelta_VER_minus_Opponent_s":
                "{:.3f}".format,
            "MeanDelta_VER_minus_Opponent_s":
                "{:.3f}".format,
            "MedianVERTyreLife":
                "{:.1f}".format,
            "MedianOpponentTyreLife":
                "{:.1f}".format,
        },
    )
)
print()
print(
    "Positive VER-minus-opponent deltas mean that Verstappen "
    "was slower on the paired laps."
)
print()

# Comparison B:
# Match exactly the same observed tyre-life values, while accepting
# that the drivers reached those tyre ages at different race phases.


def prepare_tyre_age_band(frame, driver):
    """Prepare unique integer tyre-life observations from 5 to 12."""

    selected = (
        frame.loc[
            frame["TyreLife"].between(5, 12),
            ["LapNumber", "LapTimeSec", "TyreLife"],
        ]
        .dropna()
        .sort_values("TyreLife")
        .copy()
    )

    rounded_tyre_life = selected["TyreLife"].round()

    if not np.allclose(
        selected["TyreLife"].to_numpy(dtype=float),
        rounded_tyre_life.to_numpy(dtype=float),
    ):
        raise ValueError(
            f"{driver} contains non-integer tyre-life values."
        )

    selected["TyreLifeKey"] = rounded_tyre_life.astype(int)

    if selected["TyreLifeKey"].duplicated().any():
        raise ValueError(
            f"{driver} contains duplicate observations for one tyre age."
        )

    return selected


tyre_age_band_laps = {
    driver: prepare_tyre_age_band(stint_laps, driver)
    for driver, stint_laps in final_soft_laps.items()
}

common_tyre_life_values = sorted(
    set.intersection(
        *[
            set(frame["TyreLifeKey"].tolist())
            for frame in tyre_age_band_laps.values()
        ]
    )
)

if len(common_tyre_life_values) < 4:
    raise ValueError(
        "Fewer than four common tyre-life values are available."
    )

print(
    "Common observed tyre-life values used in Comparison B:",
    common_tyre_life_values,
)
print()

ver_matched_age = tyre_age_band_laps["VER"].loc[
    tyre_age_band_laps["VER"]["TyreLifeKey"].isin(
        common_tyre_life_values
    )
].copy()

matched_tyre_age_rows = []

for opponent in ["NOR", "LEC", "HAM"]:
    opponent_matched_age = tyre_age_band_laps[opponent].loc[
        tyre_age_band_laps[opponent]["TyreLifeKey"].isin(
            common_tyre_life_values
        )
    ].copy()

    paired = (
        ver_matched_age.loc[
            :,
            [
                "TyreLifeKey",
                "LapNumber",
                "LapTimeSec",
            ],
        ]
        .merge(
            opponent_matched_age.loc[
                :,
                [
                    "TyreLifeKey",
                    "LapNumber",
                    "LapTimeSec",
                ],
            ],
            on="TyreLifeKey",
            suffixes=("_VER", f"_{opponent}"),
            how="inner",
            validate="one_to_one",
        )
        .sort_values("TyreLifeKey")
    )

    if paired["TyreLifeKey"].tolist() != common_tyre_life_values:
        raise AssertionError(
            f"Tyre-life matching failed for VER and {opponent}."
        )

    paired["DeltaSec"] = (
        paired["LapTimeSec_VER"]
        - paired[f"LapTimeSec_{opponent}"]
    )

    matched_tyre_age_rows.append(
        {
            "Opponent": opponent,
            "n": len(paired),
            "TyreLifeRange": (
                f"{int(paired['TyreLifeKey'].min())}-"
                f"{int(paired['TyreLifeKey'].max())}"
            ),
            "VERRaceLapRange": (
                f"{int(paired['LapNumber_VER'].min())}-"
                f"{int(paired['LapNumber_VER'].max())}"
            ),
            "OpponentRaceLapRange": (
                f"{int(paired[f'LapNumber_{opponent}'].min())}-"
                f"{int(paired[f'LapNumber_{opponent}'].max())}"
            ),
            "MedianDelta_VER_minus_Opponent_s":
                paired["DeltaSec"].median(),
            "MeanDelta_VER_minus_Opponent_s":
                paired["DeltaSec"].mean(),
            "VERFasterTyreAges":
                int(paired["DeltaSec"].lt(0).sum()),
            "OpponentFasterTyreAges":
                int(paired["DeltaSec"].gt(0).sum()),
        }
    )

matched_tyre_age_comparison = pd.DataFrame(
    matched_tyre_age_rows
)

print(
    "Comparison B: exact matched tyre-life values, "
    "but different race phases"
)
print(
    matched_tyre_age_comparison.to_string(
        index=False,
        formatters={
            "MedianDelta_VER_minus_Opponent_s":
                "{:.3f}".format,
            "MeanDelta_VER_minus_Opponent_s":
                "{:.3f}".format,
        },
    )
)
print()
print(
    "Positive VER-minus-opponent deltas mean that Verstappen "
    "was slower at the matched tyre ages."
)
print()

# Explicit comparison-design limitations.

comparison_design_check = pd.DataFrame(
    [
        {
            "Comparison": "Same race phase",
            "RacePhaseMatched": True,
            "TyreAgeMatched": False,
            "TrafficAssessed": False,
        },
        {
            "Comparison": "Exact matched tyre age",
            "RacePhaseMatched": False,
            "TyreAgeMatched": True,
            "TrafficAssessed": False,
        },
    ]
)

print("Comparison-design limitations:")
print(
    comparison_design_check.to_string(index=False)
)

assert not (
    comparison_design_check["RacePhaseMatched"]
    & comparison_design_check["TyreAgeMatched"]
).any()

Comparison A: same race phase and common lap numbers, but different tyre ages
Opponent  n LapRange MedianDelta_VER_minus_Opponent_s MeanDelta_VER_minus_Opponent_s  VERFasterLaps  OpponentFasterLaps MedianVERTyreLife MedianOpponentTyreLife
     NOR 12    58-70                            0.666                          0.770              1                  11              26.5                   11.5
     LEC 13    58-70                            0.706                          0.775              2                  11              26.0                   11.0
     HAM 13    58-70                            0.410                          0.319              3                  10              26.0                   11.0

Positive VER-minus-opponent deltas mean that Verstappen was slower on the paired laps.

Common observed tyre-life values used in Comparison B: [5, 6, 8, 9, 10, 11]

Comparison B: exact matched tyre-life values, but different race phases
Opponent  n TyreLifeRange VERRaceLapRang

## Cross-driver comparison interpretation

Both comparison designs show higher observed lap times for Verstappen than for
the other three drivers, although each design retains an important confounding
difference.

### Same-race-phase comparison

Across common eligible laps from 58 to 70, Verstappen records a positive median
lap-time difference against every opponent:

- 0.666 seconds per lap relative to Norris;
- 0.706 seconds per lap relative to Leclerc;
- 0.410 seconds per lap relative to Hamilton.

A positive difference means that Verstappen was slower.

He was faster on only 1 of 12 paired laps against Norris, 2 of 13 against
Leclerc, and 3 of 13 against Hamilton.

However, Verstappen's median recorded tyre life was approximately 26 laps,
compared with approximately 11 laps for the other drivers. These results
therefore describe the late-race pace difference but cannot separate relative
performance from the substantial tyre-age difference.

### Exact matched-tyre-age comparison

At the six exact tyre-life values shared by all four eligible samples,
Verstappen again records a positive median lap-time difference against every
opponent:

- 1.354 seconds per lap relative to Norris;
- 1.518 seconds per lap relative to Leclerc;
- 0.383 seconds per lap relative to Hamilton.

Verstappen was slower at all six matched tyre ages against Norris and at four
of six against both Leclerc and Hamilton.

This comparison removes the observed tyre-age difference, but it introduces a
large race-phase difference. Verstappen's matched laps occur on race laps 43 to
49, whereas the comparison laps occur on laps 58 to 64 for the other drivers.
Verstappen would ordinarily be expected to carry more fuel and is observed at
an earlier stage of race and track evolution.

The two comparisons consequently cannot produce a controlled estimate of
relative performance. Nevertheless, neither comparison provides positive
evidence that Verstappen's final soft stint delivered unusually strong pace
relative to the selected front runners.

## Stage 6 synthesis and conclusion

The targeted checks do not support promoting Verstappen's final soft stint as
a main finding.

The in-stint analysis confirms that he completed a long eligible sample with
relatively low lap-time variation. However, the fitted trend is sensitive to
the selected window, and `LapNumber` is perfectly correlated with `TyreLife`
within the stint. The slope therefore cannot separate tyre behaviour from fuel
burn, track evolution, traffic, driver management, or other changes over race
time.

The cross-driver comparisons also provide no positive evidence of unusually
strong relative pace. Verstappen records higher observed lap times than Norris,
Leclerc, and Hamilton under both comparison designs. Neither result is
controlled:

- the same-race-phase comparison leaves tyre age unmatched;
- the exact matched-tyre-age comparison leaves race phase unmatched.

The defensible conclusion is therefore limited:

> Verstappen completed a long final soft-tyre stint with relatively low
> lap-time variation, but the available evidence does not show exceptional tyre
> preservation, superior late-race pace, or a better alternative strategy.

Candidate 2 is closed as secondary descriptive context. Stage 6 does not
identify a sufficiently supported main story.

# Stage 7: Focused Leclerc-Hamilton sector comparison

Stage 6 closed the final active main-story candidate. This stage does not reopen
broad story discovery.

It completes the focused sector comparison deferred earlier under Module D. The
question is limited to whether Leclerc recorded a repeatable sector-level timing
difference relative to Hamilton during their final soft-tyre stints.

The Leclerc-Hamilton pair is justified using information independent of the
sector deltas. They were adjacent in the recorded finishing order, their
official positions were reversed only by Hamilton's penalty, and both
completed overlapping final Stint 4 soft-tyre samples. Stage 7 therefore
performs one bounded pairwise follow-up rather than searching broadly for the
largest sector difference.

The comparison uses:

- Stint 4 on the `SOFT` compound for both drivers;
- the existing green-flag eligibility rule;
- common eligible race-lap numbers;
- lap and sector times from the FastF1 lap table.

For every paired observation, the difference is calculated as:

`Leclerc time - Hamilton time`

A negative value means Leclerc recorded the lower time.

A compact sensitivity check tests whether the Sector 2 direction depends on the
first paired lap, one portion of the comparison window, or one individual lap.

This stage does not load telemetry or attempt to identify a corner-level,
technical, or causal mechanism.

In [28]:
# Compare Leclerc and Hamilton on common eligible laps from their
# final SOFT stints and run compact Sector 2 sensitivity checks.

sector_columns = [
    "Driver",
    "LapNumber",
    "Stint",
    "Compound",
    "TyreLife",
    "LapTimeSec",
    "Sector1Time",
    "Sector2Time",
    "Sector3Time",
]


def select_final_soft_laps(driver):
    """Select eligible laps from one driver's final SOFT stint."""

    selected = (
        eligible_laps.loc[
            eligible_laps["Driver"].eq(driver)
            & eligible_laps["Stint"].eq(4)
            & eligible_laps["Compound"].eq("SOFT"),
            sector_columns,
        ]
        .sort_values("LapNumber")
        .copy()
    )

    if selected.empty:
        raise ValueError(
            f"No eligible final-stint SOFT laps found for {driver}."
        )

    for sector in [
        "Sector1Time",
        "Sector2Time",
        "Sector3Time",
    ]:
        selected[f"{sector}Sec"] = (
            selected[sector].dt.total_seconds()
        )

    return selected


lec_final_soft = select_final_soft_laps("LEC")
ham_final_soft = select_final_soft_laps("HAM")

paired_sector_laps = (
    lec_final_soft[
        [
            "LapNumber",
            "TyreLife",
            "LapTimeSec",
            "Sector1TimeSec",
            "Sector2TimeSec",
            "Sector3TimeSec",
        ]
    ]
    .merge(
        ham_final_soft[
            [
                "LapNumber",
                "TyreLife",
                "LapTimeSec",
                "Sector1TimeSec",
                "Sector2TimeSec",
                "Sector3TimeSec",
            ]
        ],
        on="LapNumber",
        how="inner",
        suffixes=("_LEC", "_HAM"),
        validate="one_to_one",
    )
    .dropna()
    .sort_values("LapNumber")
    .reset_index(drop=True)
)

assert len(paired_sector_laps) == 13
assert (
    paired_sector_laps["LapNumber"]
    .astype(int)
    .tolist()
    == list(range(58, 71))
)

paired_sector_laps["LapDelta"] = (
    paired_sector_laps["LapTimeSec_LEC"]
    - paired_sector_laps["LapTimeSec_HAM"]
)

paired_sector_laps["S1Delta"] = (
    paired_sector_laps["Sector1TimeSec_LEC"]
    - paired_sector_laps["Sector1TimeSec_HAM"]
)

paired_sector_laps["S2Delta"] = (
    paired_sector_laps["Sector2TimeSec_LEC"]
    - paired_sector_laps["Sector2TimeSec_HAM"]
)

paired_sector_laps["S3Delta"] = (
    paired_sector_laps["Sector3TimeSec_LEC"]
    - paired_sector_laps["Sector3TimeSec_HAM"]
)

sector_sum_delta = (
    paired_sector_laps["S1Delta"]
    + paired_sector_laps["S2Delta"]
    + paired_sector_laps["S3Delta"]
)

sector_sum_error = (
    paired_sector_laps["LapDelta"]
    - sector_sum_delta
).abs()

assert sector_sum_error.max() < 0.01

print(
    "FOCUSED SECTOR COMPARISON: "
    "NEGATIVE DELTA MEANS LECLERC WAS FASTER"
)
print()

summary_rows = []

for measure, column in [
    ("Full lap", "LapDelta"),
    ("Sector 1", "S1Delta"),
    ("Sector 2", "S2Delta"),
    ("Sector 3", "S3Delta"),
]:
    values = paired_sector_laps[column]

    summary_rows.append(
        {
            "Measure": measure,
            "n": len(values),
            "MedianDelta_LEC_minus_HAM_s":
                values.median(),
            "LECFasterCount":
                int(values.lt(0).sum()),
            "HAMFasterCount":
                int(values.gt(0).sum()),
            "TieCount":
                int(values.eq(0).sum()),
        }
    )

sector_summary = pd.DataFrame(summary_rows)

print("Paired comparison summary:")
print(
    sector_summary.to_string(
        index=False,
        formatters={
            "MedianDelta_LEC_minus_HAM_s":
                "{:.3f}".format,
        },
    )
)
print()

print(
    "Recorded tyre-life differences, LEC minus HAM:",
    sorted(
        (
            paired_sector_laps["TyreLife_LEC"]
            - paired_sector_laps["TyreLife_HAM"]
        )
        .unique()
        .tolist()
    ),
)
print()

# Compact Sector 2 sensitivity checks.

s2 = paired_sector_laps[
    ["LapNumber", "S2Delta"]
].copy()

sensitivity_rows = []

for label, selected in [
    (
        "All paired laps",
        s2,
    ),
    (
        "Excluding lap 58",
        s2.loc[s2["LapNumber"].ne(58)],
    ),
    (
        "Early portion",
        s2.loc[s2["LapNumber"].between(58, 64)],
    ),
    (
        "Late portion",
        s2.loc[s2["LapNumber"].between(65, 70)],
    ),
]:
    sensitivity_rows.append(
        {
            "Window": label,
            "n": len(selected),
            "MedianS2Delta_s":
                selected["S2Delta"].median(),
            "LECFasterCount":
                int(selected["S2Delta"].lt(0).sum()),
            "HAMFasterCount":
                int(selected["S2Delta"].gt(0).sum()),
        }
    )

sector2_sensitivity = pd.DataFrame(
    sensitivity_rows
)

leave_one_out_medians = []

for removed_lap in s2["LapNumber"]:
    retained = s2.loc[
        s2["LapNumber"].ne(removed_lap),
        "S2Delta",
    ]

    leave_one_out_medians.append(
        retained.median()
    )

print("Sector 2 sensitivity:")
print(
    sector2_sensitivity.to_string(
        index=False,
        formatters={
            "MedianS2Delta_s": "{:.3f}".format,
        },
    )
)
print()

print(
    "Leave-one-out median range:",
    f"{min(leave_one_out_medians):.3f}",
    "to",
    f"{max(leave_one_out_medians):.3f}",
    "seconds",
)

assert sector2_sensitivity["MedianS2Delta_s"].lt(0).all()
assert all(
    median < 0
    for median in leave_one_out_medians
)

FOCUSED SECTOR COMPARISON: NEGATIVE DELTA MEANS LECLERC WAS FASTER

Paired comparison summary:
 Measure  n MedianDelta_LEC_minus_HAM_s  LECFasterCount  HAMFasterCount  TieCount
Full lap 13                      -0.201               8               5         0
Sector 1 13                       0.141               4               9         0
Sector 2 13                      -0.476              12               1         0
Sector 3 13                       0.135               4               8         1

Recorded tyre-life differences, LEC minus HAM: [0.0]

Sector 2 sensitivity:
          Window  n MedianS2Delta_s  LECFasterCount  HAMFasterCount
 All paired laps 13          -0.476              12               1
Excluding lap 58 12          -0.383              11               1
   Early portion  7          -0.666               6               1
    Late portion  6          -0.281               6               0

Leave-one-out median range: -0.505 to -0.383 seconds


## Stage 7 interpretation and conclusion

The focused comparison identifies one repeatable but limited sector-level
pattern.

Across 13 common eligible final-stint laps, Leclerc records the lower Sector 2
time on 12 laps, with a median advantage of approximately 0.476 seconds. The
recorded tyre-life difference is zero on every paired lap, so tyre age is
aligned within this comparison.

The difference is concentrated in Sector 2:

- Hamilton generally records the lower time in Sector 1;
- Leclerc records a persistent advantage in Sector 2;
- Hamilton generally records the lower time in Sector 3;
- Leclerc's median full-lap advantage is smaller and less consistent than the
  Sector 2 difference.

The Sector 2 direction remains unchanged after:

- excluding the first paired lap;
- dividing the sample into early and late portions;
- removing each paired lap individually.

The leave-one-out median remains between approximately -0.505 and -0.383
seconds. The result is therefore not produced by one isolated observation.

However, matching race-lap number and recorded tyre life does not control
traffic, track position, driver management, or every difference in race
circumstances. Sector timing also does not identify a technical mechanism.

The defensible finding is limited to:

> Leclerc recorded a repeatable Sector 2 timing advantage over Hamilton during
> their paired final-stint laps.

This is retained as a secondary descriptive observation. It is not promoted as
a technical explanation, a consequential race mechanism, or a main standalone
story.

Stage 7 does not identify a sufficiently supported main story.

# Stage 8: Final editorial decision and project closure

This stage records the final disposition of the findings evaluated in Stages 4
through 7. It does not introduce new data, calculations, external research, or
candidate searches.

The editorial question is whether any finding has enough evidentiary strength,
originality, sporting significance, and explanatory value to support the
substantive standalone Hungarian Grand Prix article originally sought.

A correct or repeatable observation is not promoted merely because it is
measurable. It must also provide a meaningful and defensible race story.

## Final candidate dispositions

### Penalty and classification reconstruction

**Final disposition: `SECONDARY FINDING ONLY`**

Two unserved five-second penalties produced five changes between the last
recorded running order and the official final classification:

- Hamilton moved from recorded fourth to classified fifth;
- Leclerc moved from recorded fifth to classified fourth;
- Bearman moved from recorded seventeenth to classified nineteenth;
- Albon and Sainz each gained one classified position.

The reconstruction is direct and internally consistent, but it is a routine
consequence of applying the official penalties to the finishing order. It does
not reveal a hidden mechanism, change the winner or podium, or provide an
original main-story result.

### Verstappen's final soft stint

**Final disposition: `SECONDARY DESCRIPTIVE CONTEXT`**

Verstappen completed a long final soft-tyre stint with relatively low
eligible-lap variation.

Targeted validation did not establish exceptional tyre preservation, unusually
strong relative pace, or a better alternative strategy. The in-stint slope was
confounded by race progression, while the cross-driver comparisons could not
match both tyre age and race phase.

The candidate was therefore tested and closed rather than left awaiting further
validation.

### Leclerc-Hamilton Sector 2 pattern

**Final disposition: `SECONDARY FINDING ONLY`**

Across 13 common eligible final-stint laps with equal recorded tyre age,
Leclerc recorded the lower Sector 2 time on 12 laps, with a median advantage of
approximately 0.476 seconds.

The direction persisted across the early and late portions of the sample and
under every leave-one-out check. However:

- the full-lap difference was smaller and less consistent;
- traffic, driver management, and other race circumstances were not controlled;
- no technical mechanism was established;
- the pattern did not produce an on-track position change.

The result is repeatable at sector-timing resolution but remains limited in
sporting and editorial significance.

## Final Hungarian Grand Prix decision

**`NO MAIN-ARTICLE-STRENGTH STORY IDENTIFIED`**

The notebook evaluated several candidate families, including:

- differences between the recorded running order and final classification;
- stint-level pace summaries;
- tyre-life and lap-time trends;
- Virtual Safety Car and pit-stop sequences;
- broad pace-rank differences;
- Verstappen's extended final soft stint;
- a focused Leclerc-Hamilton sector comparison.

The final result is not based on a failure to find any measurable pattern.
Several observations were valid, but none combined all of the qualities needed
for the intended article:

- the penalty reconstruction was correct but routine;
- Verstappen's stint did not survive targeted validation as an exceptional
  performance or strategy result;
- the Leclerc-Hamilton Sector 2 pattern was repeatable but lacked a verified
  mechanism and meaningful race consequence.

Promoting any of these observations into a substantive race story would require
adding interpretation beyond what the evidence supports.

Story discovery for the 2026 Hungarian Grand Prix therefore closes without a
main standalone article. The notebook remains a reproducible analytical audit
containing valid secondary observations, rejected candidates, and documented
limitations.

## Lessons for the next race

1. Start with sporting consequence. Prioritize incidents, strategy decisions,
   position battles, or performance reversals that materially affected the
   race outcome.

2. Separate measurement from significance. A repeatable numerical difference
   is not automatically an important or original story.

3. Evaluate confounding conditions early. Race phase, tyre age, fuel burn,
   traffic, pit-cycle state, and track status should be checked before
   investing in detailed candidate analysis.

4. Use broad screens only to generate bounded questions. Do not search every
   driver, lap, and sector for the largest isolated difference.

5. Require a defensible path from observation to interpretation. Sector or
   stint timing can locate a pattern, but it does not by itself establish a
   technical or strategic mechanism.

6. Close weak candidates once targeted validation fails. Do not continue adding
   analyses merely to preserve an initially interesting pattern.

7. Treat a no-story result as valid. Stopping without an article is preferable
   to constructing a narrative that exceeds the available evidence.